<a href="https://colab.research.google.com/github/mbrennan5/LSTM-TREND/blob/claude%2Fplan-session-VX3Ru/LSTM_VOLUME_FAMILY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, gc, shutil, numpy as np, pandas as pd, yfinance as yf
from datetime import datetime
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Input
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import warnings
from google.colab import drive

warnings.filterwarnings('ignore')

# ==============================================================================
# BLOCK 1: ENVIRONMENT & CORE CONFIGURATION
# ==============================================================================
drive.mount('/content/drive', force_remount=True)

LOCAL_DB = '/content/FRICTION_MASTER_DB.parquet'
DRIVE_DB_PATH = '/content/drive/MyDrive/backtest_results/FRICTION_MASTER_DB.parquet'
OUTPUT_DRIVE_DIR = '/content/drive/MyDrive/judicial_results/'

if not os.path.exists(OUTPUT_DRIVE_DIR): os.makedirs(OUTPUT_DRIVE_DIR)
if not os.path.exists(LOCAL_DB) and os.path.exists(DRIVE_DB_PATH):
    print("🚚 Syncing Master Database to Local Runtime...")
    shutil.copy(DRIVE_DB_PATH, LOCAL_DB)

print("\n--- 🧠 JUDICIAL BRAIN SELECTION ---")
print("1. DIRECTION (DIR_val) [Auto-GRU]")
print("2. EASE (EASE_val) [Auto-LSTM]")
print("3. EXP (EXP_val) [Auto-LSTM]")
print("4. ALL BRAINS (Full Sequence)")
brain_choice = input("Select Brain (1/2/3/4): ")

GLOBAL_MODE = False
BRAIN_TAG = ""

if brain_choice == '4':
    BRAINS_TO_RUN = ['DIRECTION', 'EASE', 'EXP']
    print("\n--- ⚖️ AGGREGATION LOGIC ---")
    print("1. BRAIN SEGREGATED (Individual CSV per target)")
    print("2. GLOBAL CONSENSUS (One unified CSV for collective performance)")
    agg_choice = input("Select Aggregation Mode (1/2): ")
    if agg_choice == '2':
        GLOBAL_MODE = True
        BRAIN_TAG = "GLOBAL_CONSENSUS"
    else:
        BRAIN_TAG = "ALL_BRAINS_SEGREGATED"
else:
    selected = {'1': 'DIRECTION', '2': 'EASE', '3': 'EXP'}.get(brain_choice, 'DIRECTION')
    BRAINS_TO_RUN = [selected]
    BRAIN_TAG = selected

ITERATIONS = int(input(f"\nEnter Iterations per Brain: "))
SYMBOLS_PER_RUN = int(input("Enter Symbols per Iteration: ") or 60)
HORIZON = 1 # [2026-02-23] Default

use_locked = input("\nUtilize the 'Locked Core' feature set? (y/n): ").lower() == 'y'
CORE_LOCKED_LIST = [
    'LENS_10_slope_amivest', 'LENS_90_slope_amivest', 'LENS_90_z_amivest',
   ] if use_locked else []

# ==============================================================================
# BLOCK 2: THE PHYSICS ENGINE (Updated Naming Convention)
# ==============================================================================
FAMILY_MAP = {
    'fi': ('Force', 'Energy'), 'emv': ('Ease of Movement', 'Energy'),
    'fve': ('Finite Vol Elements', 'Flow'), 'cmf': ('Money Flow', 'Flow'),
    'obv': ('Accumulation', 'Flow'), 'pvt': ('Conviction', 'Flow'),
    'vwap_dev': ('Displacement', 'Context'), 'mtsi': ('Mean Reversion', 'Context'),
    'mobius_bsp': ('Boundary Range', 'Context'), 'liquidity_ratio': ('Friction', 'Efficiency'),
    'amivest': ('Efficiency', 'Efficiency'), 'pvo': ('Volume Momentum', 'Energy'),
    'rel_vol': ('Volume Shock', 'Energy'), 'kvo': ('Klinger', 'Flow'),
    'harlin_spike': ('Pressure', 'Energy'), 'harlin_osc': ('Pressure', 'Flow'),
    'obv_osc': ('Accumulation', 'Energy'), 'ratio_force_friction': ('Sovereign', 'Efficiency'),
    'ratio_vel_friction': ('Sovereign', 'Efficiency'), 'ratio_inst_squeeze': ('Sovereign', 'Context')
}

def get_feature_meta(feat_name):
    seed = "unknown"
    for k in FAMILY_MAP.keys():
        if f"_{k}" in feat_name: seed = k; break
    family, m_type = FAMILY_MAP.get(seed, ("Misc", "Misc"))
    return seed, family, m_type

def generate_heavy_physics(df):
    h, l, c, v = df['High'].squeeze(), df['Low'].squeeze(), df['Close'].squeeze(), df['Volume'].squeeze()
    tp = (h + l + c) / 3
    seeds = pd.DataFrame(index=df.index)
    # 1. CORE FRICTION
    seeds['liquidity_ratio'] = v.rolling(30, min_periods=1).sum() / (c.pct_change().abs().rolling(30, min_periods=1).sum() + 1e-9)
    seeds['amivest'] = v / (c.pct_change().abs() + 1e-9)
    seeds['cmf'] = (((c-l)-(h-c))/(h-l+1e-9)*v).rolling(30, min_periods=1).sum() / (v.rolling(30, min_periods=1).sum()+1e-9)
    seeds['fi'] = (c.diff(1) * v).ewm(span=30).mean()
    seeds['rel_vol'] = v / (v.rolling(30, min_periods=1).mean() + 1e-9)

    # REPAIRED TMF: Using np.minimum/maximum for True Range stability
    seeds['tmf'] = ((((c - np.minimum(l, c.shift(1))) / (np.maximum(h, c.shift(1)) - np.minimum(l, c.shift(1)) + 1e-9)) * 2 - 1) * v).ewm(span=30, min_periods=1).mean() / (v.ewm(span=30, min_periods=1).mean() + 1e-9)

    # 2. KINEMATIC FLOW
    seeds['emv'] = ((h+l)/2 - (h.shift(1)+l.shift(1))/2) / ((v / (h-l+1e-9)) + 1e-9)
    seeds['fve'] = (np.where(tp > tp.shift(1), v, -v) / (v.rolling(30, min_periods=1).mean() * 30 + 1e-9)).rolling(30, min_periods=1).sum()
    seeds['pvo'] = (v.ewm(span=12).mean() - v.ewm(span=30).mean()) / (v.ewm(span=30).mean() + 1e-9)

    # 3. EXOTICS
    seeds['harlin_spike'] = (c.diff() / (h-l+1e-9)).rolling(5, min_periods=1).mean()
    seeds['harlin_osc'] = seeds['harlin_spike'].ewm(span=10).mean() - seeds['harlin_spike'].ewm(span=30).mean()
    seeds['mobius_bsp'] = (c - l.rolling(30, min_periods=1).min()) / (h.rolling(30, min_periods=1).max() - l.rolling(30, min_periods=1).min() + 1e-9)
    seeds['vwap_dev'] = (c - ((tp * v).cumsum() / (v.cumsum() + 1e-9))) / (c.rolling(30, min_periods=1).std() + 1e-9)

    # 4. HYBRID FRICTION
    seeds['ratio_force_friction'] = seeds['fi'] / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_vel_friction'] = (c.diff() / (c.shift(1) + 1e-9)) / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_inst_squeeze'] = seeds['cmf'] / (seeds['pvo'].abs() + 1e-9)
    seeds['ratio_inst_squeezeV2'] = seeds['tmf'] / (seeds['pvo'].abs() + 1e-9)
    seeds['kaufman_vol_hybrid'] = (c.diff(10).abs() / (c.diff(1).abs().rolling(10, min_periods=1).sum() + 1e-9)) * seeds['rel_vol']
    seeds['hurst_vol_persistence'] = get_hurst_fast(v, window=100).fillna(0.5)

    # 5. CUMULATIVE
    seeds_cum = pd.DataFrame(index=df.index)
    seeds_cum['pvt'] = ((c.diff() / (c.shift(1) + 1e-9)) * v).cumsum()
    seeds_cum['obv'] = (np.sign(c.diff()) * v).fillna(0).cumsum()
    seeds['obv_osc'] = seeds_cum['obv'].ewm(span=10).mean() - seeds_cum['obv'].ewm(span=30).mean()


    expanded = []
    for w in [10, 90]:
        z = (seeds - seeds.rolling(w).mean()) / (seeds.rolling(w).std() + 1e-9)
        for col in z.columns:
            expanded.append(pd.Series(z[col], name=f'LENS_{w}_z_{col}'))
            expanded.append(pd.Series(z[col].diff(3), name=f'LENS_{w}_slope_{col}'))
    for w in [10, 30, 90]:
        reset = (seeds_cum - seeds_cum.shift(w)) / (seeds_cum.shift(w).rolling(w).std() + 1e-9)
        for col in reset.columns:
            expanded.append(pd.Series(reset[col], name=f'LENS_{w}_reset_{col}'))
    return pd.concat(expanded, axis=1).ffill().dropna()

# ==============================================================================
# BLOCK 3: AUDIT ENGINE & ARCHITECTURE LOCKS
# ==============================================================================
def run_residual_audit(brain_name, target_col, master_df, locked_list):
    # Forced Architecture Rule
    CURRENT_ARCH = 'GRU' if brain_name == 'DIRECTION' else 'LSTM'

    all_feats = [c for c in master_df.columns if 'LENS_' in c]
    X_raw = RobustScaler().fit_transform(master_df[all_feats])
    y_raw = master_df[target_col].values
    pca = PCA(n_components=10).fit(X_raw)
    uniqueness_raw = np.abs(pca.components_).sum(axis=0)
    mid = len(X_raw) // 2
    stability_raw = 1 / (1 + (np.abs(X_raw[:mid].mean(axis=0) - X_raw[mid:].mean(axis=0)) / (X_raw.std(axis=0) + 1e-9)))

    X_s, y_s = [], []
    for i in range(len(X_raw)-10):
        X_s.append(X_raw[i:i+10]); y_s.append(y_raw[i+10])
    X_s, y_s = np.array(X_s), np.array(y_s)

    model = Sequential([Input(shape=(10, X_raw.shape[1]))])
    if CURRENT_ARCH == 'GRU': model.add(GRU(64))
    else: model.add(LSTM(64))

    model.add(Dense(1, activation='sigmoid' if brain_name == 'DIRECTION' else 'linear'))
    model.compile(optimizer='adam', loss='binary_crossentropy' if brain_name == 'DIRECTION' else 'mse')
    model.fit(X_s, y_s, epochs=4, batch_size=256, verbose=0)

    impact_raw = []
    # Weighted for Ease of Capture Predictability
    for i in tqdm(range(len(all_feats)), desc=f"⚖️ Scoring ({CURRENT_ARCH})", leave=False):
        save = X_s[:500, :, i].copy(); np.random.shuffle(X_s[:500, :, i])
        impact_raw.append(model.evaluate(X_s[:500], y_s[:500], verbose=0)); X_s[:500, :, i] = save

    res = pd.DataFrame({'Feature': all_feats, 'Impact_R': impact_raw, 'Stability_R': stability_raw, 'Uniqueness_R': uniqueness_raw})
    i_n, s_n, u_n = [(res[c] - res[c].min()) / (res[c].max() - res[c].min() + 1e-9) for c in ['Impact_R', 'Stability_R', 'Uniqueness_R']]
    res['Finalist_Tiebreaker'], res['Pillar_Score'], res['U_Norm_Temp'] = (s_n*0.4 + u_n*0.4 + i_n*0.2), (i_n*0.5 + s_n*0.3 + u_n*0.2), u_n
    return res, pd.DataFrame(X_raw, columns=all_feats).corr()

def process_nuanced_logic(df, corr_matrix, locked_list):
    df[['Seed_Name', 'Family', 'Type']] = df['Feature'].apply(lambda x: pd.Series(get_feature_meta(x)))
    df['Judicial_Tier'], df['Judicial_Reason'] = "CULLED", "Insufficient Alpha Depth"

    # 1. PRIMARY OVERRIDE: Identify and seal LOCKED features immediately
    survivors = {f: ("LOCKED CORE PILLAR", "User Core Selection") for f in locked_list if f in df['Feature'].values}

    # 2. ALPHA LEADER PASS (Only for non-locked features)
    eligible = df[~df['Feature'].isin(survivors.keys())].sort_values('Pillar_Score', ascending=False)
    top_seeds = eligible.groupby('Seed_Name')['Pillar_Score'].mean().sort_values(ascending=False).head(2).index
    for seed in top_seeds:
        variants = [f"LENS_90_z_{seed}", f"LENS_90_slope_{seed}", f"LENS_10_slope_{seed}", f"LENS_10_z_{seed}"]
        for v in variants:
            # Check if it exists AND isn't already in survivors (to keep the LOCKED label)
            if v in df['Feature'].values and v not in survivors and len(survivors) < 25:
                survivors[v] = ("PILLAR LEADER (ALPHA)", "Top-Tier Architecture")

    # 3. DIVERSITY PASS (Only for non-locked/non-alpha features)
    pool = df.sort_values(['U_Norm_Temp', 'Pillar_Score'], ascending=False)
    alpha_cutoff = eligible['Pillar_Score'].quantile(0.7) if not eligible.empty else 0

    for _, row in pool.iterrows():
        if len(survivors) >= 35 or row['Feature'] in survivors: continue

        active = [f for f in survivors.keys() if f in corr_matrix.columns]
        max_c = abs(corr_matrix.loc[row['Feature'], active]).max() if active else 0

        # We don't cull the LOCKED features, but we cull others that are too close to LOCKED features
        if max_c > 0.85: continue

        if row['U_Norm_Temp'] >= 0.6 or row['Pillar_Score'] > alpha_cutoff:
            label = "CHAOS HONOREE" if row['U_Norm_Temp'] > 0.8 else "CONSENSUS EDGE"
            survivors[row['Feature']] = (label, f"Diversity Pass (r={max_c:.2f})")

    # FINAL LABEL MAPPING
    def apply_labels(row):
        if row['Feature'] in survivors:
            return survivors[row['Feature']][0], survivors[row['Feature']][1]
        return "CULLED", row['Judicial_Reason']

    df[['Judicial_Tier', 'Judicial_Reason']] = df.apply(lambda r: pd.Series(apply_labels(r)), axis=1)
    return df

# ==============================================================================
# BLOCK 4: EXECUTION & DYNAMIC EXPORT
# ==============================================================================
if os.path.exists(LOCAL_DB):
    db = pd.read_parquet(LOCAL_DB)
    TEMP_COLLECTION = []

    for CURRENT_BRAIN in BRAINS_TO_RUN:
        print(f"\n🚀 STARTING {CURRENT_BRAIN}...")
        brain_iterations, brain_corr = [], None
        target_col = {'DIRECTION': 'DIR_val', 'EASE': 'EASE_val', 'EXP': 'EXP_val'}.get(CURRENT_BRAIN)

        for i in range(1, ITERATIONS + 1):
            all_batches = []
            selected_symbols = db['symbol'].value_counts().sample(n=min(SYMBOLS_PER_RUN, len(db['symbol'].unique()))).index

            for s in tqdm(selected_symbols, desc=f"Iteration {i}", leave=False):
                try:
                    df = yf.download(s, start="2021-02-19", progress=False, auto_adjust=True)
                    if df.empty: continue
                    physics, s_db = generate_heavy_physics(df), db[db['symbol'] == s].copy()
                    s_db['T_FINAL'] = s_db[target_col].shift(-HORIZON)
                    all_batches.append(physics.join(s_db, how='inner').dropna())
                except: continue

            if all_batches:
                res, corr = run_residual_audit(CURRENT_BRAIN, 'T_FINAL', pd.concat(all_batches), CORE_LOCKED_LIST)
                brain_iterations.append(res)
                brain_corr = corr if brain_corr is None else (brain_corr + corr) / 2
            gc.collect()

        if brain_iterations:
            df_final = pd.concat(brain_iterations).groupby('Feature').mean().reset_index()
            df_final['Brain'] = CURRENT_BRAIN
            TEMP_COLLECTION.append((df_final, brain_corr))

    FINAL_LIST = []
    if GLOBAL_MODE:
        all_data = pd.concat([x[0] for x in TEMP_COLLECTION])
        glob_avg = all_data.groupby('Feature').mean().reset_index()
        glob_avg['Brain'], avg_corr = "GLOBAL_CONSENSUS", pd.concat([x[1] for x in TEMP_COLLECTION]).groupby(level=0).mean()
        FINAL_LIST.append(process_nuanced_logic(glob_avg, avg_corr, CORE_LOCKED_LIST))
    else:
        for b_df, b_corr in TEMP_COLLECTION:
            FINAL_LIST.append(process_nuanced_logic(b_df, b_corr, CORE_LOCKED_LIST))

    if FINAL_LIST:
        mega = pd.concat(FINAL_LIST).sort_values(['Brain', 'Judicial_Tier', 'Pillar_Score'], ascending=[True, True, False])
        final_path = f"{OUTPUT_DRIVE_DIR}JUDICIAL_MASTER_{BRAIN_TAG}.csv"
        mega.to_csv(final_path, index=False)
        print(f"\n🏁 COMPLETE | Saved: {final_path}")
        display(mega[['Brain', 'Feature', 'Judicial_Tier', 'Pillar_Score', 'Judicial_Reason']].head(35))

Mounted at /content/drive

--- 🧠 JUDICIAL BRAIN SELECTION ---
1. DIRECTION (DIR_val) [Auto-GRU]
2. EASE (EASE_val) [Auto-LSTM]
3. EXP (EXP_val) [Auto-LSTM]
4. ALL BRAINS (Full Sequence)
Select Brain (1/2/3/4): 1

Enter Iterations per Brain: 1
Enter Symbols per Iteration: 1


KeyboardInterrupt: Interrupted by user

In [ ]:
import os, gc, shutil, numpy as np, pandas as pd, yfinance as yf
from datetime import datetime
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Input
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import warnings
from google.colab import drive

warnings.filterwarnings('ignore')

# ==============================================================================
# ## TEST IDENTIFICATION: Sovereign Titan Engine v3.5 (The Kinematic Revision)
# ## MANDATE: Stationarity + Culled Feature Reporting
# ==============================================================================

drive.mount('/content/drive', force_remount=True)

TEST_NAME = "Sovereign_Titan_v3.5_Kinematic_Revision"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")
OUTPUT_DRIVE_DIR = f'/content/drive/MyDrive/judicial_results/{TEST_NAME}_{TIMESTAMP}/'
if not os.path.exists(OUTPUT_DRIVE_DIR): os.makedirs(OUTPUT_DRIVE_DIR)

print(f"\n--- 🧬 {TEST_NAME.replace('_', ' ')} ---")
brain_choice = input("Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): ")
BRAINS_TO_RUN = ['DIRECTION', 'EASE', 'EXP'] if brain_choice == '4' else [{'1':'DIRECTION','2':'EASE','3':'EXP'}[brain_choice]]

ITERATIONS = int(input("Iterations per Stage (Def 5): ") or 5)
SYMBOLS_PER_RUN = int(input("Symbols per Iteration (Def 55): ") or 55)
HORIZON = 1
TARGET_OUTPUT_COUNT = 19

# ==============================================================================
# ## BLOCK 2: PHYSICS (OSC Z-SPACE + NON-OSC KINEMATIC)
# ==============================================================================
DIMENSION_MAP = {
    'fi': 'Energy', 'emv': 'Energy', 'fve': 'Flow', 'cmf': 'Flow',
    'obv': 'Flow', 'pvt': 'Flow', 'vwap_dev': 'Context', 'mtsi': 'Context',
    'mobius_bsp': 'Context', 'liquidity_ratio': 'Efficiency',
    'amivest': 'Efficiency', 'pvo': 'Energy', 'rel_vol': 'Energy',
    'kvo': 'Flow', 'harlin_spike': 'Energy', 'harlin_osc': 'Flow',
    'obv_osc': 'Energy', 'ratio_force_friction': 'Sovereign',
    'ratio_vel_friction': 'Sovereign', 'ratio_inst_squeeze': 'Sovereign'
}

def get_feature_meta(feat_name):
    seed = "unknown"
    for k in DIMENSION_MAP.keys():
        if f"_{k}" in feat_name: seed = k; break
    return seed, DIMENSION_MAP.get(seed, "Misc")

def generate_heavy_physics(df):
    h, l, c, v = df['High'].squeeze(), df['Low'].squeeze(), df['Close'].squeeze(), df['Volume'].squeeze()
    tp = (h + l + c) / 3

    seeds = pd.DataFrame(index=df.index)
    seeds['liquidity_ratio'] = v.rolling(30).sum() / (c.pct_change().abs().rolling(30).sum() + 1e-9)
    seeds['amivest'] = v / (c.pct_change().abs() + 1e-9)
    seeds['cmf'] = (((c-l)-(h-c))/(h-l+1e-9)*v).rolling(30).sum() / (v.rolling(30).sum()+1e-9)
    seeds['fi'] = (c.diff(1) * v).ewm(span=30).mean()
    seeds['emv'] = ((h+l)/2 - (h.shift(1)+l.shift(1))/2) / ((v / (h-l+1e-9)) + 1e-9)
    seeds['fve'] = (np.where(tp > tp.shift(1), v, -v) / (v.rolling(30).mean() * 30 + 1e-9)).rolling(30).sum()
    seeds['kvo'] = pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=34).mean() - \
                   pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=55).mean()
    seeds['pvo'] = (v.ewm(span=12).mean() - v.ewm(span=30).mean()) / (v.ewm(span=30).mean() + 1e-9)
    seeds['rel_vol'] = v / (v.rolling(30).mean() + 1e-9)
    seeds['mtsi'] = (c - ((tp * v).rolling(2).sum() / (v.rolling(2).sum() + 1e-9))).ewm(span=3).mean()
    seeds['harlin_spike'] = (c.diff() / (h-l+1e-9)).rolling(5).mean()
    seeds['harlin_osc'] = seeds['harlin_spike'].ewm(span=10).mean() - seeds['harlin_spike'].ewm(span=30).mean()
    seeds['mobius_bsp'] = (c - l.rolling(30).min()) / (h.rolling(30).max() - l.rolling(30).min() + 1e-9)
    vwap_val = (tp * v).cumsum() / (v.cumsum() + 1e-9)
    seeds['vwap_dev'] = (c - vwap_val) / (c.rolling(30).std() + 1e-9)
    seeds['ratio_force_friction'] = seeds['fi'] / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_vel_friction'] = (c.diff() / (c.shift(1) + 1e-9)) / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_inst_squeeze'] = seeds['cmf'] / (seeds['pvo'].abs() + 1e-9)

    seeds_cum = pd.DataFrame(index=df.index)
    seeds_cum['pvt'] = ((c.diff() / (c.shift(1) + 1e-9)) * v).cumsum()
    seeds_cum['obv'] = (np.sign(c.diff()) * v).fillna(0).cumsum()

    expanded = []

    # TREATMENT A: OSCILLATORS (Z + 1st & 2nd Deriv) @ 10, 90
    for w in [10, 90]:
        z = (seeds - seeds.rolling(w).mean()) / (seeds.rolling(w).std() + 1e-9)
        for col in z.columns:
            expanded.append(pd.Series(z[col], name=f'LENS_{w}_z_{col}'))
            slope = z[col].diff(3)
            expanded.append(pd.Series(slope, name=f'LENS_{w}_slope_{col}'))
            expanded.append(pd.Series(slope.diff(2), name=f'LENS_{w}_accel_{col}'))

    # TREATMENT B: NON-OSCILLATORS (Kinematic Stationarity) @ 10, 30, 60
    for w in [10, 30, 60]:
        for col in ['pvt', 'obv']:
            floor = abs(seeds_cum[col].min()) + 1
            stat_diff = np.log(seeds_cum[col] + floor).diff().fillna(0)
            expanded.append(pd.Series(stat_diff.rolling(w).sum(), name=f'LENS_{w}_KIN_state_{col}'))
            vel = stat_diff.rolling(w).mean()
            expanded.append(pd.Series(vel, name=f'LENS_{w}_KIN_vel_{col}'))
            expanded.append(pd.Series(vel.diff(3), name=f'LENS_{w}_KIN_accel_{col}'))

    return pd.concat(expanded, axis=1).ffill().dropna()

# ==============================================================================
# ## BLOCK 3: JUDICIAL SELECTION & CULLING LOGIC
# ==============================================================================
def run_residual_audit(brain_name, target_col, master_df):
    ARCH = 'GRU' if brain_name == 'DIRECTION' else 'LSTM'
    all_feats = [c for c in master_df.columns if 'LENS_' in c]
    scaler = RobustScaler()
    X_raw = scaler.fit_transform(master_df[all_feats])
    y_raw = master_df[target_col].values

    n = len(X_raw)
    s1, s2, s3 = X_raw[:n//3], X_raw[n//3:2*n//3], X_raw[2*n//3:]
    drift = np.abs(s1.mean(axis=0) - s2.mean(axis=0)) + np.abs(s2.mean(axis=0) - s3.mean(axis=0))
    s_raw = 1 / (1 + drift)

    model = Sequential([Input(shape=(1, X_raw.shape[1]))])
    model.add(GRU(64) if ARCH == 'GRU' else LSTM(64))
    model.add(Dense(1, activation='sigmoid' if brain_name == 'DIRECTION' else 'linear'))
    model.compile(optimizer='adam', loss='binary_crossentropy' if brain_name == 'DIRECTION' else 'mse')
    model.fit(X_raw.reshape(-1, 1, X_raw.shape[1]), y_raw, epochs=2, batch_size=1024, verbose=0)

    X_test = X_raw[:1000].reshape(-1, 1, X_raw.shape[1])
    base_err = model.evaluate(X_test, y_raw[:1000], verbose=0)
    i_raw = []
    for i in range(len(all_feats)):
        save = X_test[:, :, i].copy()
        np.random.shuffle(X_test[:, :, i])
        i_raw.append(abs(model.evaluate(X_test, y_raw[:1000], verbose=0) - base_err))
        X_test[:, :, i] = save

    pca = PCA(n_components=min(15, len(all_feats))).fit(X_raw)
    u_raw = np.abs(pca.components_).sum(axis=0)

    res = pd.DataFrame({'Feature': all_feats, 'I_raw': i_raw, 'S_raw': s_raw, 'U_raw': u_raw})
    for c in ['I_raw', 'S_raw', 'U_raw']:
        res[f'{c[0]}_Norm'] = (res[c] - res[c].min()) / (res[c].max() - res[c].min() + 1e-9)

    return res, pd.DataFrame(X_raw, columns=all_feats).corr()

# ==============================================================================
# ## BLOCK 4: EXECUTION & REPORTING
# ==============================================================================
if os.path.exists('/content/FRICTION_MASTER_DB.parquet'):
    db = pd.read_parquet('/content/FRICTION_MASTER_DB.parquet')

    for BRAIN in BRAINS_TO_RUN:
        target_col = {'DIRECTION': 'DIR_val', 'EASE': 'EASE_val', 'EXP': 'EXP_val'}[BRAIN]
        print(f"\n🚀 BRAIN: {BRAIN} | {TEST_NAME}")

        all_batches = []
        syms = np.random.choice(db['symbol'].unique(), min(SYMBOLS_PER_RUN, len(db['symbol'].unique())), replace=False)
        for s in tqdm(syms, desc="Harvesting"):
            try:
                raw = yf.download(s, start="2022-01-01", progress=False, auto_adjust=True)
                p = generate_heavy_physics(raw)
                s_db = db[db['symbol'] == s].copy()
                s_db['T_FINAL'] = s_db[target_col].shift(-HORIZON)
                all_batches.append(p.join(s_db, how='inner').dropna(subset=['T_FINAL']))
            except: continue

        master_df = pd.concat(all_batches)
        res, corr = run_residual_audit(BRAIN, 'T_FINAL', master_df)

        kill_threshold = res['S_Norm'].quantile(0.15)
        selection, culled_report = [], []

        for stage in range(1, TARGET_OUTPUT_COUNT + 1):
            is_pillar = stage <= 10
            def judicial_scorer(row):
                if row['Feature'] in selection: return -99
                # CULLING REASON: Stability Kill
                if row['S_Norm'] < kill_threshold:
                    culled_report.append({'Feature': row['Feature'], 'Reason': 'STABILITY_KILL'})
                    return -1

                seed, _ = get_feature_meta(row['Feature'])
                # CULLING REASON: Rule of 3 (Seed Dominance)
                if sum(1 for f in selection if get_feature_meta(f)[0] == seed) >= 3:
                    culled_report.append({'Feature': row['Feature'], 'Reason': 'SEED_DOMINANCE'})
                    return -1

                # CULLING REASON: Correlation Gate
                if selection and abs(corr.loc[row['Feature'], selection]).max() > 0.95:
                    culled_report.append({'Feature': row['Feature'], 'Reason': 'CORR_GATE'})
                    return -1

                if is_pillar: return (row['I_Norm']*0.7 + row['S_Norm']*0.2 + row['U_Norm']*0.1)
                else: return (row['U_Norm']*0.7 + row['I_Norm']*0.2 + row['S_Norm']*0.1)

            res['Score'] = res.apply(judicial_scorer, axis=1)
            winner = res.sort_values('Score', ascending=False).iloc[0]
            selection.append(winner['Feature'])

        # ## GENERATE CULLED SUMMARY
        culled_df = pd.DataFrame(culled_report).drop_duplicates('Feature')
        culled_counts = culled_df['Reason'].value_counts()

        # ## PCA RECONSTRUCTION
        full_pool = [c for c in master_df.columns if 'LENS_' in c]
        selected_data = RobustScaler().fit_transform(master_df[selection])
        recon_score = (np.sum(PCA().fit(selected_data).explained_variance_) /
                       np.sum(PCA().fit(RobustScaler().fit_transform(master_df[full_pool])).explained_variance_)) * 100

        print(f"\n📊 FAMILY RECONSTRUCTION: {recon_score:.2f}%")
        print(f"🛑 CULLED FEATURES: {len(culled_df)} total features vaporized.")
        print(culled_counts)

        # ## SAVE REQUISITE FILES
        final_rep = res[res['Feature'].isin(selection)].copy()
        final_rep['Tier'] = ["PILLAR" if i < 10 else "CHAOS" for i in range(len(selection))]
        final_rep.to_csv(os.path.join(OUTPUT_DRIVE_DIR, f"{BRAIN}_results.csv"), index=False)
        culled_df.to_csv(os.path.join(OUTPUT_DRIVE_DIR, f"{BRAIN}_CULLED_REPORT.csv"), index=False)

        display(final_rep[['Feature', 'Tier', 'I_Norm', 'S_Norm', 'U_Norm']])

Mounted at /content/drive

--- 🧬 Sovereign Titan v3.6.1 Hybrid Objective ---


Harvesting DIRECTION:   0%|          | 0/1 [00:00<?, ?it/s]


📊 DIRECTION RECONSTRUCTION: 0.00%


,Feature,Tier,I_Norm,S_Norm,U_Norm
40,LENS_10_slope_vwap_dev,PILLAR,0.143851,0.987220,0.045558
45,LENS_10_z_ratio_vel_friction,PILLAR,0.051618,0.995786,0.177141
46,LENS_10_slope_ratio_vel_friction,PILLAR,0.034967,0.998406,0.237707
55,LENS_90_slope_amivest,PILLAR,0.825789,0.815906,0.743970
56,LENS_90_accel_amivest,PILLAR,0.094658,0.980308,0.806965
63,LENS_90_z_emv,PILLAR,0.093512,0.962948,0.336173
64,LENS_90_slope_emv,PILLAR,0.008079,0.985021,0.261862
65,LENS_90_accel_emv,PILLAR,0.058524,0.972674,0.387284
66,LENS_90_z_fve,PILLAR,0.062997,0.836511,0.176985
95,LENS_90_accel_ratio_force_friction,PILLAR,0.058604,0.998208,0.157351


In [ ]:
import os, gc, shutil, numpy as np, pandas as pd, yfinance as yf
from datetime import datetime
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Input
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import warnings
from google.colab import drive

warnings.filterwarnings('ignore')

# ==============================================================================
# BLOCK 1: CONFIG & ENVIRONMENT
# ==============================================================================
drive.mount('/content/drive', force_remount=True)

LOCAL_DB = '/content/FRICTION_MASTER_DB.parquet'
DRIVE_DB_PATH = '/content/drive/MyDrive/backtest_results/FRICTION_MASTER_DB.parquet'

W_QUARTER = 60
W_MONTH = 30
W_FAST = 10
HORIZON = 1    # [2026-02-23] Default

# ==============================================================================
# BLOCK 2: METADATA & PHYSICS (PRESERVATION MODE)
# ==============================================================================
DIMENSION_MAP = {
    'fi': 'Energy', 'emv': 'Energy', 'fve': 'Flow', 'cmf': 'Flow',
    'obv': 'Flow', 'pvt': 'Flow', 'vwap_dev': 'Context', 'mtsi': 'Context',
    'mobius_bsp': 'Context', 'liquidity_ratio': 'Efficiency',
    'amivest': 'Efficiency', 'pvo': 'Energy', 'rel_vol': 'Energy',
    'kvo': 'Flow', 'harlin_spike': 'Energy', 'harlin_osc': 'Flow',
    'obv_osc': 'Energy', 'ratio_force_friction': 'Sovereign',
    'ratio_vel_friction': 'Sovereign', 'ratio_inst_squeeze': 'Sovereign'
}

def get_feature_meta(feat_name):
    seed = "unknown"
    for k in DIMENSION_MAP.keys():
        if f"_{k}" in feat_name: seed = k; break
    return seed, DIMENSION_MAP.get(seed, "Misc")

def generate_heavy_physics(df):
    h, l, c, v = df['High'].squeeze(), df['Low'].squeeze(), df['Close'].squeeze(), df['Volume'].squeeze()
    tp = (h + l + c) / 3

    # --- OSCILLATOR SEEDS (UNTOUCHED) ---
    seeds = pd.DataFrame(index=df.index)
    seeds['liquidity_ratio'] = v.rolling(30).sum() / (c.pct_change().abs().rolling(30).sum() + 1e-9)
    seeds['amivest'] = v / (c.pct_change().abs() + 1e-9)
    seeds['cmf'] = (((c-l)-(h-c))/(h-l+1e-9)*v).rolling(30).sum() / (v.rolling(30).sum()+1e-9)
    seeds['fi'] = (c.diff(1) * v).ewm(span=30).mean()
    seeds['emv'] = ((h+l)/2 - (h.shift(1)+l.shift(1))/2) / ((v / (h-l+1e-9)) + 1e-9)
    seeds['fve'] = (np.where(tp > tp.shift(1), v, -v) / (v.rolling(30).mean() * 30 + 1e-9)).rolling(30).sum()
    seeds['kvo'] = pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=34).mean() - \
                   pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=55).mean()
    seeds['pvo'] = (v.ewm(span=12).mean() - v.ewm(span=30).mean()) / (v.ewm(span=30).mean() + 1e-9)
    seeds['rel_vol'] = v / (v.rolling(30).mean() + 1e-9)
    seeds['mtsi'] = (c - ((tp * v).rolling(2).sum() / (v.rolling(2).sum() + 1e-9))).ewm(span=3).mean()
    seeds['harlin_spike'] = (c.diff() / (h-l+1e-9)).rolling(5).mean()
    seeds['harlin_osc'] = seeds['harlin_spike'].ewm(span=10).mean() - seeds['harlin_spike'].ewm(span=30).mean()
    seeds['mobius_bsp'] = (c - l.rolling(30).min()) / (h.rolling(30).max() - l.rolling(30).min() + 1e-9)
    vwap_val = (tp * v).cumsum() / (v.cumsum() + 1e-9)
    seeds['vwap_dev'] = (c - vwap_val) / (c.rolling(30).std() + 1e-9)
    seeds['ratio_force_friction'] = seeds['fi'] / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_vel_friction'] = (c.diff() / (c.shift(1) + 1e-9)) / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_inst_squeeze'] = seeds['cmf'] / (seeds['pvo'].abs() + 1e-9)

    # --- NON-OSCILLATOR SEEDS (KINEMATIC TREATMENT) ---
    seeds_cum = pd.DataFrame(index=df.index)
    seeds_cum['pvt'] = ((c.diff() / (c.shift(1) + 1e-9)) * v).cumsum()
    seeds_cum['obv'] = (np.sign(c.diff()) * v).fillna(0).cumsum()
    seeds['obv_osc'] = seeds_cum['obv'].ewm(span=10).mean() - seeds_cum['obv'].ewm(span=30).mean()

    expanded = []

    # 1. Standard Lensing (Oscillators)
    for w in [W_MONTH, W_QUARTER]:
        z = (seeds - seeds.rolling(w).mean()) / (seeds.rolling(w).std() + 1e-9)
        for col in z.columns:
            expanded.append(pd.Series(z[col], name=f'LENS_{w}_z_{col}'))
            expanded.append(pd.Series(z[col].diff(3), name=f'LENS_{w}_slope_{col}'))
            expanded.append(pd.Series(z[col].diff(3).diff(2), name=f'LENS_{w}_accel_{col}'))

    # 2. Kinematic Lensing (Non-Oscillators) @ 10, 30, 60
    for w in [W_FAST, W_MONTH, W_QUARTER]:
        for col in ['pvt', 'obv']:
            floor = abs(seeds_cum[col].min()) + 1
            stat_diff = np.log(seeds_cum[col] + floor).diff().fillna(0)

            # State, Velocity, Acceleration
            expanded.append(pd.Series(stat_diff.rolling(w).sum(), name=f'LENS_{w}_KIN_state_{col}'))
            vel = stat_diff.rolling(w).mean()
            expanded.append(pd.Series(vel, name=f'LENS_{w}_KIN_vel_{col}'))
            expanded.append(pd.Series(vel.diff(3), name=f'LENS_{w}_KIN_accel_{col}'))

    return pd.concat(expanded, axis=1).ffill().dropna()

# ==============================================================================
# BLOCK 3: JUDICIAL SELECTION (70/20/10 RULE)
# ==============================================================================
def run_residual_audit(brain_name, target_col, master_df):
    all_feats = [c for c in master_df.columns if 'LENS_' in c]
    scaler = RobustScaler().fit(master_df[all_feats].iloc[:len(master_df)//2])
    X_raw = scaler.transform(master_df[all_feats])
    y_raw = master_df[target_col].values

    # Stability Split
    n = len(X_raw)
    s1, s2, s3 = X_raw[:n//3], X_raw[n//3:2*n//3], X_raw[2*n//3:]
    drift = np.abs(s1.mean(axis=0) - s2.mean(axis=0)) + np.abs(s2.mean(axis=0) - s3.mean(axis=0))
    s_raw = 1 / (1 + drift)

    # Quick Permutation Impact
    model = Sequential([Input(shape=(1, X_raw.shape[1]))])
    model.add(LSTM(32)) # Standardized probe
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_raw.reshape(-1, 1, X_raw.shape[1]), y_raw, epochs=1, batch_size=1024, verbose=0)

    X_test = X_raw[:500].reshape(-1, 1, X_raw.shape[1])
    base_err = model.evaluate(X_test, y_raw[:500], verbose=0)
    i_raw = []
    for i in range(len(all_feats)):
        save = X_test[:, :, i].copy()
        np.random.shuffle(X_test[:, :, i])
        i_raw.append(abs(model.evaluate(X_test, y_raw[:500], verbose=0) - base_err))
        X_test[:, :, i] = save

    res = pd.DataFrame({'Feature': all_feats, 'I_raw': i_raw, 'S_raw': s_raw})
    res['I_Norm'] = (res['I_raw'] - res['I_raw'].min()) / (res['I_raw'].max() - res['I_raw'].min() + 1e-9)
    res['S_Norm'] = (res['S_raw'] - res['S_raw'].min()) / (res['S_raw'].max() - res['S_raw'].min() + 1e-9)

    # PCA Uniqueness
    pca = PCA(n_components=min(10, len(all_feats))).fit(X_raw)
    res['U_Norm'] = np.abs(pca.components_).sum(axis=0)
    res['U_Norm'] = (res['U_Norm'] - res['U_Norm'].min()) / (res['U_Norm'].max() - res['U_Norm'].min() + 1e-9)

    return res, pd.DataFrame(X_raw, columns=all_feats).corr()

# ==============================================================================
# BLOCK 4: EXECUTION
# ==============================================================================
if os.path.exists(LOCAL_DB):
    db = pd.read_parquet(LOCAL_DB)
    target_col = 'DIR_val' # Example: Direction Brain

    print(f"🚀 RUNNING SOVEREIGN TITAN v4.12")
    all_batches = []
    syms = np.random.choice(db['symbol'].unique(), 40, replace=False)
    for s in tqdm(syms, desc="Harvesting"):
        try:
            raw = yf.download(s, start="2022-01-01", progress=False, auto_adjust=True)
            p = generate_heavy_physics(raw)
            s_db = db[db['symbol'] == s].copy()
            s_db['T_FINAL'] = s_db[target_col].shift(-HORIZON)
            all_batches.append(p.join(s_db, how='inner').dropna(subset=['T_FINAL']))
        except: continue

    master_df = pd.concat(all_batches)
    res, corr = run_residual_audit('DIRECTION', 'T_FINAL', master_df)

    # Judicial Gates
    kill_gate = res['S_Norm'].quantile(0.15)
    pillar_gate = res['S_Norm'].quantile(0.25)

    selection = []
    for stage in range(1, 20):
        is_pillar = stage <= 10
        def scorer(row):
            if row['Feature'] in selection: return -99
            if row['S_Norm'] < kill_gate: return -1
            if is_pillar and row['S_Norm'] < pillar_gate: return -1
            seed, _ = get_feature_meta(row['Feature'])
            if sum(1 for f in selection if get_feature_meta(f)[0] == seed) >= 3: return -1
            if selection and abs(corr.loc[row['Feature'], selection]).max() > 0.95: return -1

            if is_pillar: return (row['I_Norm']*0.7 + row['S_Norm']*0.2 + row['U_Norm']*0.1)
            else: return (row['U_Norm']*0.7 + row['I_Norm']*0.2 + row['S_Norm']*0.1)

        res['Score'] = res.apply(scorer, axis=1)
        winner = res.sort_values('Score', ascending=False).iloc[0]
        selection.append(winner['Feature'])

    display(res[res['Feature'].isin(selection)][['Feature', 'I_Norm', 'S_Norm', 'U_Norm']])

Mounted at /content/drive
🚀 RUNNING SOVEREIGN TITAN v4.12


Harvesting:   0%|          | 0/40 [00:00<?, ?it/s]

,Feature,I_Norm,S_Norm,U_Norm
4,LENS_30_slope_amivest,0.783071,0.984239,0.161511
44,LENS_30_accel_ratio_force_friction,0.030401,0.989214,0.105103
45,LENS_30_z_ratio_vel_friction,0.018269,0.873291,0.138767
50,LENS_30_accel_ratio_inst_squeeze,0.380152,0.986004,0.163159
58,LENS_60_slope_amivest,1.000000,0.967631,0.318416
59,LENS_60_accel_amivest,0.494267,0.983437,0.145233
65,LENS_60_accel_fi,0.103652,0.995984,0.116683
95,LENS_60_accel_vwap_dev,0.084309,0.979446,0.077877
97,LENS_60_slope_ratio_force_friction,0.039154,0.959149,0.093518
100,LENS_60_slope_ratio_vel_friction,0.015017,0.989231,0.148291


In [ ]:
import os, gc, shutil, numpy as np, pandas as pd, yfinance as yf
from datetime import datetime
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Input
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn import set_config
from tqdm.auto import tqdm
import warnings
from google.colab import drive

# ## MANDATE: Identity Preservation & Pure Weighting Logic
set_config(transform_output="pandas")
warnings.filterwarnings('ignore')

# ==============================================================================
# ## DRIVE & PATH CONFIGURATION
# ==============================================================================
drive.mount('/content/drive', force_remount=True)

DRIVE_DB_PATH = '/content/drive/MyDrive/backtest_results/FRICTION_MASTER_DB.parquet'
TEST_NAME = "Sovereign_Titan_v3.6.4_Pure_Merit"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")
OUTPUT_DRIVE_DIR = f'/content/drive/MyDrive/judicial_results/{TEST_NAME}_{TIMESTAMP}/'

if not os.path.exists(OUTPUT_DRIVE_DIR): os.makedirs(OUTPUT_DRIVE_DIR)

# ==============================================================================
# ## BLOCK 1: CORE PARAMS
# ==============================================================================
print(f"\n--- 🧬 {TEST_NAME.replace('_', ' ')} ---")
brain_choice = input("Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): ")
BRAINS_TO_RUN = ['DIRECTION', 'EASE', 'EXP'] if brain_choice == '4' else [{'1':'DIRECTION','2':'EASE','3':'EXP'}[brain_choice]]

ITERATIONS = int(input("Iterations per Stage (Def 5): ") or 5)
SYMBOLS_PER_RUN = int(input("Symbols per Iteration (Def 66): ") or 66)
HORIZON = 1
TARGET_OUTPUT_COUNT = 19

# ==============================================================================
# ## BLOCK 2: KINEMATIC PHYSICS (v3.5 REVISION)
# ==============================================================================
def generate_heavy_physics(df):
    h, l, c, v = df['High'].squeeze(), df['Low'].squeeze(), df['Close'].squeeze(), df['Volume'].squeeze()
    tp = (h + l + c) / 3
    seeds = pd.DataFrame(index=df.index)
    seeds['liquidity_ratio'] = v.rolling(30).sum() / (c.pct_change().abs().rolling(30).sum() + 1e-9)
    seeds['amivest'] = v / (c.pct_change().abs() + 1e-9)
    seeds['cmf'] = (((c-l)-(h-c))/(h-l+1e-9)*v).rolling(30).sum() / (v.rolling(30).sum()+1e-9)
    seeds['fi'] = (c.diff(1) * v).ewm(span=30).mean()
    seeds['emv'] = ((h+l)/2 - (h.shift(1)+l.shift(1))/2) / ((v / (h-l+1e-9)) + 1e-9)
    seeds['fve'] = (np.where(tp > tp.shift(1), v, -v) / (v.rolling(30).mean() * 30 + 1e-9)).rolling(30).sum()
    seeds['kvo'] = pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=34).mean() - \
                   pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=55).mean()
    seeds['pvo'] = (v.ewm(span=12).mean() - v.ewm(span=30).mean()) / (v.ewm(span=30).mean() + 1e-9)
    seeds['rel_vol'] = v / (v.rolling(30).mean() + 1e-9)
    seeds['mtsi'] = (c - ((tp * v).rolling(2).sum() / (v.rolling(2).sum() + 1e-9))).ewm(span=3).mean()
    seeds['harlin_spike'] = (c.diff() / (h-l+1e-9)).rolling(5).mean()
    seeds['harlin_osc'] = seeds['harlin_spike'].ewm(span=10).mean() - seeds['harlin_spike'].ewm(span=30).mean()
    seeds['mobius_bsp'] = (c - l.rolling(30).min()) / (h.rolling(30).max() - l.rolling(30).min() + 1e-9)
    vwap_val = (tp * v).cumsum() / (v.cumsum() + 1e-9)
    seeds['vwap_dev'] = (c - vwap_val) / (c.rolling(30).std() + 1e-9)
    seeds['ratio_force_friction'] = seeds['fi'] / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_vel_friction'] = (c.diff() / (c.shift(1) + 1e-9)) / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_inst_squeeze'] = seeds['cmf'] / (seeds['pvo'].abs() + 1e-9)

    seeds_cum = pd.DataFrame(index=df.index)
    seeds_cum['pvt'] = ((c.diff() / (c.shift(1) + 1e-9)) * v).cumsum()
    seeds_cum['obv'] = (np.sign(c.diff()) * v).fillna(0).cumsum()

    expanded = []
    for w in [10, 90]:
        z = (seeds - seeds.rolling(w).mean()) / (seeds.rolling(w).std() + 1e-9)
        for col in z.columns:
            expanded.append(pd.Series(z[col], name=f'LENS_{w}_z_{col}'))
            slope = z[col].diff(3)
            expanded.append(pd.Series(slope, name=f'LENS_{w}_slope_{col}'))
            expanded.append(pd.Series(slope.diff(2), name=f'LENS_{w}_accel_{col}'))

    for w in [10, 30, 60]:
        for col in ['pvt', 'obv']:
            floor = abs(seeds_cum[col].min()) + 1
            stat_diff = np.log(seeds_cum[col] + floor).diff().fillna(0)
            expanded.append(pd.Series(stat_diff.rolling(w).sum(), name=f'LENS_{w}_KIN_state_{col}'))
            vel = stat_diff.rolling(w).mean()
            expanded.append(pd.Series(vel, name=f'LENS_{w}_KIN_vel_{col}'))
            expanded.append(pd.Series(vel.diff(3), name=f'LENS_{w}_KIN_accel_{col}'))
    return pd.concat(expanded, axis=1).ffill().dropna()

# ==============================================================================
# ## BLOCK 3: JUDICIAL AUDIT ENGINE (SHAPE FIXED)
# ==============================================================================
def run_residual_audit(brain_name, target_col, master_df):
    ARCH = 'GRU' if brain_name == 'DIRECTION' else 'LSTM'
    all_feats = [c for c in master_df.columns if 'LENS_' in c]

    scaler = RobustScaler()
    X_df = scaler.fit_transform(master_df[all_feats])
    X_vals = X_df.values
    y_raw = master_df[target_col].values

    n = len(X_vals)
    s1, s2, s3 = X_vals[:n//3], X_vals[n//3:2*n//3], X_vals[2*n//3:]
    drift = np.abs(s1.mean(axis=0) - s2.mean(axis=0)) + np.abs(s2.mean(axis=0) - s3.mean(axis=0))
    s_raw = 1 / (1 + drift)

    loss_fn = 'binary_crossentropy' if brain_name == 'DIRECTION' else tf.keras.losses.Huber()
    out_act = 'sigmoid' if brain_name == 'DIRECTION' else 'linear'

    model = Sequential([Input(shape=(1, X_vals.shape[1]))])
    model.add(GRU(64) if ARCH == 'GRU' else LSTM(64))
    model.add(Dense(1, activation=out_act))
    model.compile(optimizer='adam', loss=loss_fn)
    model.fit(X_vals.reshape(-1, 1, X_vals.shape[1]), y_raw, epochs=2, batch_size=1024, verbose=0)

    X_test = X_vals[:1000].reshape(-1, 1, X_vals.shape[1])
    base_err = model.evaluate(X_test, y_raw[:1000], verbose=0)
    i_raw = []
    for i in range(len(all_feats)):
        save = X_test[:, :, i].copy()
        np.random.shuffle(X_test[:, :, i])
        i_raw.append(abs(model.evaluate(X_test, y_raw[:1000], verbose=0) - base_err))
        X_test[:, :, i] = save

    pca = PCA(n_components=min(20, len(all_feats))).fit(X_vals)
    u_raw = np.abs(pca.components_).sum(axis=0)

    res = pd.DataFrame({'Feature': all_feats, 'I_raw': i_raw, 'S_raw': s_raw, 'U_raw': u_raw})
    for c in ['I_raw', 'S_raw', 'U_raw']:
        res[f'{c[0]}_Norm'] = (res[c] - res[c].min()) / (res[c].max() - res[c].min() + 1e-9)
    return res, pd.DataFrame(X_vals, columns=all_feats).corr()

# ==============================================================================
# ## BLOCK 4: HARVESTING & PURE MERIT SELECTION
# ==============================================================================
if os.path.exists(DRIVE_DB_PATH):
    db = pd.read_parquet(DRIVE_DB_PATH)
    for BRAIN in BRAINS_TO_RUN:
        target_col = {'DIRECTION': 'DIR_val', 'EASE': 'EASE_val', 'EXP': 'EXP_val'}[BRAIN]
        all_batches = []
        syms = np.random.choice(db['symbol'].unique(), min(SYMBOLS_PER_RUN, len(db['symbol'].unique())), replace=False)

        for s in tqdm(syms, desc=f"Harvesting {BRAIN}"):
            try:
                raw = yf.download(s, start="2022-01-01", progress=False, auto_adjust=True)
                if raw.empty: continue
                if isinstance(raw.columns, pd.MultiIndex): raw.columns = [f"{col[0]}" for col in raw.columns]

                p = generate_heavy_physics(raw)
                s_db = db[db['symbol'] == s].copy()
                p.index = pd.to_datetime(p.index); s_db.index = pd.to_datetime(s_db.index)

                s_db['T_FINAL'] = s_db[target_col].shift(-HORIZON)
                joined = p.join(s_db, how='inner')
                joined['T_FINAL'] = joined['T_FINAL'].ffill()
                joined = joined.dropna(subset=['T_FINAL'])
                if not joined.empty: all_batches.append(joined)
            except: continue

        if not all_batches: print(f"❌ No data for {BRAIN}"); continue
        master_df = pd.concat(all_batches)
        res, corr = run_residual_audit(BRAIN, 'T_FINAL', master_df)

        selection = []
        for stage in range(1, TARGET_OUTPUT_COUNT + 1):
            is_pillar = stage <= 10
            def judicial_scorer(row):
                if row['Feature'] in selection: return -999
                # PILLAR: 0.70 Impact | CHAOS: 0.70 Uniqueness
                if is_pillar:
                    return (row['I_Norm'] * 0.70) + (row['S_Norm'] * 0.15) + (row['U_Norm'] * 0.15)
                else:
                    return (row['U_Norm'] * 0.70) + (row['I_Norm'] * 0.15) + (row['S_Norm'] * 0.15)

            res['Score'] = res.apply(judicial_scorer, axis=1)
            winner = res.sort_values('Score', ascending=False).iloc[0]
            selection.append(winner['Feature'])

        # Final Reconstruction Logic
        full_pool = [c for c in master_df.columns if 'LENS_' in c]
        selected_data = RobustScaler().fit_transform(master_df[selection])
        recon_score = (np.sum(PCA().fit(selected_data.values).explained_variance_) /
                       np.sum(PCA().fit(RobustScaler().fit_transform(master_df[full_pool]).values).explained_variance_)) * 100

        print(f"\n📊 {BRAIN} RECONSTRUCTION: {recon_score:.2f}%")
        final_rep = res[res['Feature'].isin(selection)].copy()
        final_rep['Selection_Order'] = final_rep['Feature'].apply(lambda x: selection.index(x))
        final_rep = final_rep.sort_values('Selection_Order')
        final_rep['Tier'] = ["PILLAR" if i < 10 else "CHAOS" for i in range(len(selection))]

        final_rep.to_csv(os.path.join(OUTPUT_DRIVE_DIR, f"{BRAIN}_results.csv"), index=False)
        display(final_rep[['Feature', 'Tier', 'I_Norm', 'S_Norm', 'U_Norm']])

Mounted at /content/drive

--- 🧬 Sovereign Titan v3.6.4 Pure Merit ---
Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): 1
Iterations per Stage (Def 5): 1
Symbols per Iteration (Def 66): 1


Harvesting DIRECTION:   0%|          | 0/1 [00:00<?, ?it/s]


📊 DIRECTION RECONSTRUCTION: 90.98%


,Feature,Tier,I_Norm,S_Norm,U_Norm
100,LENS_90_slope_ratio_inst_squeeze,PILLAR,1.000000,7.218243e-01,0.879195
55,LENS_90_slope_amivest,PILLAR,0.927275,9.496794e-01,0.789459
116,LENS_60_KIN_accel_pvt,PILLAR,0.494972,7.652833e-01,0.858890
4,LENS_10_slope_amivest,PILLAR,0.453182,9.811510e-01,0.523490
101,LENS_90_accel_ratio_inst_squeeze,PILLAR,0.391824,9.493715e-01,0.750144
110,LENS_30_KIN_accel_pvt,PILLAR,0.292465,9.481355e-01,0.963886
95,LENS_90_accel_ratio_force_friction,PILLAR,0.346158,9.706729e-01,0.501667
56,LENS_90_accel_amivest,PILLAR,0.224322,9.902791e-01,0.836593
104,LENS_10_KIN_accel_pvt,PILLAR,0.142310,9.455973e-01,0.851590
62,LENS_90_accel_fi,PILLAR,0.220808,9.758539e-01,0.446056


In [ ]:
import os, gc, numpy as np, pandas as pd, yfinance as yf
from datetime import datetime
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import warnings
from google.colab import drive

# --- INITIALIZATION ---
warnings.filterwarnings('ignore')
drive.mount('/content/drive', force_remount=True)

DRIVE_DB_PATH = '/content/drive/MyDrive/backtest_results/FRICTION_MASTER_DB.parquet'
TEST_NAME = "Sovereign_Titan_v3.9.26_Agg_Fix"
OUTPUT_DRIVE_DIR = f'/content/drive/MyDrive/judicial_results/{TEST_NAME}/'
if not os.path.exists(OUTPUT_DRIVE_DIR): os.makedirs(OUTPUT_DRIVE_DIR)

# ==============================================================================
# ## BLOCK 1: PHYSICS ENGINE (VECTORIZED)
# ==============================================================================
def get_hurst_fast(series, window=100):
    returns = series.pct_change().dropna()
    std = returns.rolling(window).std()
    r_s = (returns.rolling(window).max() - returns.rolling(window).min()) / (std + 1e-9)
    return (np.log(r_s + 1e-9) / np.log(window)).fillna(0.5)

def generate_heavy_physics(df):
    if len(df) < 110: return pd.DataFrame()
    h, l, c, v = df['High'].squeeze(), df['Low'].squeeze(), df['Close'].squeeze(), df['Volume'].squeeze()
    tp = (h + l + c) / 3
    seeds = pd.DataFrame(index=df.index)

    seeds['liquidity_ratio'] = v.rolling(30).sum() / (c.pct_change().abs().rolling(30).sum() + 1e-9)
    seeds['amivest'] = v / (c.pct_change().abs() + 1e-9)
    seeds['cmf'] = (((c-l)-(h-c))/(h-l+1e-9)*v).rolling(30).sum() / (v.rolling(30).sum()+1e-9)
    seeds['fi'] = (c.diff(1) * v).ewm(span=30).mean()
    seeds['emv'] = ((h+l)/2 - (h.shift(1)+l.shift(1))/2) / ((v / (h-l+1e-9)) + 1e-9)
    seeds['fve'] = (np.where(tp > tp.shift(1), v, -v) / (v.rolling(30).mean() * 30 + 1e-9)).rolling(30).sum()
    seeds['kvo'] = pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=34).mean() - \
                   pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=55).mean()
    seeds['pvo'] = (v.ewm(span=12).mean() - v.ewm(span=30).mean()) / (v.ewm(span=30).mean() + 1e-9)
    seeds['rel_vol'] = v / (v.rolling(30).mean() + 1e-9)
    seeds['mtsi'] = (c - ((tp * v).rolling(2).sum() / (v.rolling(2).sum() + 1e-9))).ewm(span=3).mean()
    seeds['harlin_spike'] = (c.diff() / (h-l+1e-9)).rolling(5).mean()
    seeds['harlin_osc'] = seeds['harlin_spike'].ewm(span=10).mean() - seeds['harlin_spike'].ewm(span=30).mean()
    seeds['mobius_bsp'] = (c - l.rolling(30).min()) / (h.rolling(30).max() - l.rolling(30).min() + 1e-9)
    seeds['vwap_dev'] = (c - ((tp * v).cumsum() / (v.cumsum() + 1e-9))) / (c.rolling(30).std() + 1e-9)
    seeds['ratio_force_friction'] = seeds['fi'] / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_vel_friction'] = (c.diff() / (c.shift(1) + 1e-9)) / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_inst_squeeze'] = seeds['cmf'] / (seeds['pvo'].abs() + 1e-9)

    seeds_cum = pd.DataFrame(index=df.index)
    seeds_cum['pvt'] = ((c.diff() / (c.shift(1) + 1e-9)) * v).cumsum()
    seeds_cum['obv'] = (np.sign(c.diff()) * v).fillna(0).cumsum()
    seeds['obv_osc'] = seeds_cum['obv'].ewm(span=10).mean() - seeds_cum['obv'].ewm(span=30).mean()
    seeds['hurst'] = get_hurst_fast(c, window=100)

    expanded = []
    for w in [10, 90]:
        z = (seeds - seeds.rolling(w).mean()) / (seeds.rolling(w).std() + 1e-9)
        for col in z.columns:
            expanded.append(pd.Series(z[col], name=f'LENS_{w}_z_{col}'))
            slope = z[col].diff(3); expanded.append(pd.Series(slope, name=f'LENS_{w}_slope_{col}'))
            expanded.append(pd.Series(slope.diff(2), name=f'LENS_{w}_accel_{col}'))
    for w in [10, 30, 60]:
        for col in ['pvt', 'obv']:
            floor = abs(seeds_cum[col].min()) + 1
            stat_diff = np.log(seeds_cum[col] + floor).diff().fillna(0)
            expanded.append(pd.Series(stat_diff.rolling(w).sum(), name=f'LENS_{w}_KIN_state_{col}'))
            vel = stat_diff.rolling(w).mean(); expanded.append(pd.Series(vel, name=f'LENS_{w}_KIN_vel_{col}'))
            expanded.append(pd.Series(vel.diff(3), name=f'LENS_{w}_KIN_accel_{col}'))
    return pd.concat(expanded, axis=1).dropna()

# ==============================================================================
# ## BLOCK 2: JUDICIAL AUDIT ENGINE
# ==============================================================================
def run_judicial_audit(brain_name, target_col, master_df, audit_epochs=100):
    all_feats = [c for c in master_df.columns if 'LENS_' in c]
    scaler = RobustScaler()
    X_raw_2d = np.asarray(scaler.fit_transform(master_df[all_feats]))

    # Brain Mandate: GRU for DIR, LSTM for EASE/EXP
    if brain_name == 'DIRECTION':
        y_raw = (master_df[target_col] > 0).astype(int).values
        model_layer, loss, act = GRU(64), 'binary_crossentropy', 'sigmoid'
    else:
        y_raw = master_df[target_col].values
        model_layer, loss, act = LSTM(64), tf.keras.losses.Huber(), 'linear'

    X_3d = X_raw_2d.reshape(-1, 1, X_raw_2d.shape[1])
    model = Sequential([Input(shape=(1, X_raw_2d.shape[1])), model_layer, Dropout(0.2), Dense(1, activation=act)])
    model.compile(optimizer=Adam(0.001), loss=loss)

    es = EarlyStopping(monitor='loss', patience=15, restore_best_weights=True)
    model.fit(X_3d, y_raw, epochs=audit_epochs, batch_size=4096, verbose=0, callbacks=[es])

    n_test = min(2000, len(X_raw_2d))
    X_test_3d, y_test = X_3d[:n_test], y_raw[:n_test]
    base_err = model.evaluate(X_test_3d, y_test, verbose=0)

    i_raw = []
    for i in tqdm(range(len(all_feats)), desc=f"Auditing {brain_name}", leave=False):
        X_p = np.copy(X_test_3d)
        X_p[:, 0, i] = np.random.permutation(X_p[:, 0, i])
        i_raw.append(max(0, abs(model.evaluate(X_p, y_test, verbose=0) - base_err)))

    mid = n_test // 2
    s_raw = 1 / (1 + (np.abs(X_raw_2d[:mid].mean(axis=0) - X_raw_2d[mid:n_test].mean(axis=0))))
    u_raw = np.abs(PCA(n_components=min(15, len(all_feats))).fit(X_raw_2d).components_).sum(axis=0)

    res = pd.DataFrame({'Feature': all_feats, 'I_raw': i_raw, 'S_raw': s_raw, 'U_raw': u_raw})
    for c in ['I_raw', 'S_raw', 'U_raw']:
        res[f'{c[0]}_Norm'] = (res[c] - res[c].min()) / (res[c].max() - res[c].min() + 1e-9)

    res['Family'] = res['Feature'].str.split('_z_|_slope_|_accel_|_vel_|_state_').str[-1]
    res['Pillar_Score'] = (res['I_Norm']*0.4) + (res['S_Norm']*0.5) + (res['U_Norm']*0.1)

    def define_arch(row):
        if row['I_Norm'] > 0.7 and row['S_Norm'] > 0.7: return "Alpha King"
        if row['U_Norm'] > 0.8: return "Chaos Agent"
        if row['S_Norm'] > 0.85: return "The Anchor"
        return "Support Gear"
    res['Archetype'] = res.apply(define_arch, axis=1)
    return res

# ==============================================================================
# ## BLOCK 3: THE TOTAL AUDIT EXECUTION
# ==============================================================================
if os.path.exists(DRIVE_DB_PATH):
    db = pd.read_parquet(DRIVE_DB_PATH)
    db.index = pd.to_datetime(db.index).tz_localize(None).normalize()

    brain_choice = input("Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): ")
    MODE_TOGGLE = input("Select Mode (1: Recursive / 2: Shared): ")
    ITERATIONS = int(input("Iterations (Def 5): ") or 5)
    SYMBOLS_PER_RUN = int(input("Symbols (Def 66): ") or 66)
    AUDIT_EPOCHS = int(input("Audit Epochs (Def 100): ") or 100)

    BRAINS_TO_RUN = ['DIRECTION', 'EASE', 'EXP'] if brain_choice == '4' else [{'1':'DIRECTION','2':'EASE','3':'EXP'}[brain_choice]]
    all_final_ledger_data = []

    for BRAIN in BRAINS_TO_RUN:
        target_col, brain_iteration_results = {'DIRECTION': 'DIR_val', 'EASE': 'EASE_val', 'EXP': 'EXP_val'}[BRAIN], []

        if MODE_TOGGLE == '2':
            print(f"\n🚀 PRE-CACHING {BRAIN} SHARED PHYSICS...")
            all_syms = np.random.choice(db['symbol'].unique(), min(SYMBOLS_PER_RUN, len(db['symbol'].unique())), replace=False)
            raw_bulk = yf.download(list(all_syms), start="2024-01-01", end="2025-12-31", group_by='ticker', progress=False)
            batches = []
            for s in tqdm(all_syms):
                try:
                    df_s = raw_bulk[s] if len(all_syms) > 1 else raw_bulk
                    p = generate_heavy_physics(df_s)
                    s_db = db[db['symbol'] == s].copy()
                    s_db['T_FINAL'] = s_db[target_col].shift(-1)
                    joined = p.join(s_db[['T_FINAL']], how='inner').dropna()
                    if not joined.empty: batches.append(joined)
                except: continue
            master_df = pd.concat(batches)

        all_potential_features = [c for c in master_df.columns if 'LENS_' in c]
        X_scaled_full = RobustScaler().fit_transform(master_df[all_potential_features])

        for it in range(ITERATIONS):
            print(f"🔄 {BRAIN} | AUDIT {it+1}/{ITERATIONS}")
            res = run_judicial_audit(BRAIN, 'T_FINAL', master_df, audit_epochs=AUDIT_EPOCHS)

            selection = []
            family_counts = {}

            # --- STEP 1: ALPHA PILLARS (Suite 4 Rule) ---
            potential_pillars = res.sort_values('Pillar_Score', ascending=False)
            for _, row in potential_pillars.iterrows():
                if len(selection) >= 10: break
                fam = row['Family']
                if family_counts.get(fam, 0) < 3:
                    selection.append({'Feature': row['Feature'], 'Selection_Order': len(selection)+1, 'Role': 'ALPHA PILLAR'})
                    family_counts[fam] = family_counts.get(fam, 0) + 1

            # --- STEP 2: CHAOS AGENTS (Suite 4 Rule) ---
            while len(selection) < 19:
                cur_names = [m['Feature'] for m in selection]
                cur_idx = [all_potential_features.index(f) for f in cur_names]
                exhausted_fams = [f for f, count in family_counts.items() if count >= 3]
                rem_df = res[~res['Feature'].isin(cur_names) & ~res['Family'].isin(exhausted_fams)].copy()
                if rem_df.empty: break

                cur_var = np.sum(PCA(n_components=min(len(cur_names), 10)).fit(X_scaled_full[:, cur_idx]).explained_variance_)
                gain_map = {f: np.sum(PCA(n_components=min(len(cur_idx)+1, 10)).fit(X_scaled_full[:, cur_idx + [all_potential_features.index(f)]]).explained_variance_) - cur_var for f in rem_df['Feature']}
                g_min, g_max = min(gain_map.values()), max(gain_map.values())
                rem_df['Chaos_Score'] = rem_df.apply(lambda r: (r['U_Norm']*0.7) + (((gain_map.get(r['Feature'],0)-g_min)/(g_max-g_min+1e-9))*0.3), axis=1)
                best_c = rem_df.sort_values('Chaos_Score', ascending=False).iloc[0]
                selection.append({'Feature': best_c['Feature'], 'Selection_Order': len(selection)+1, 'Role': 'CHAOS AGENT'})
                family_counts[best_c['Family']] = family_counts.get(best_c['Family'], 0) + 1

            brain_iteration_results.append(res.merge(pd.DataFrame(selection), on='Feature', how='left'))
            gc.collect(); tf.keras.backend.clear_session()

        # --- SAFE AGGREGATION (Fixed for Pandas Strictness) ---
        raw_combined = pd.concat(brain_iteration_results)
        numeric_cols = ['I_Norm', 'S_Norm', 'U_Norm', 'Pillar_Score', 'Selection_Order']

        # Groupby numeric columns only
        summary_num = raw_combined.groupby('Feature')[numeric_cols].mean().reset_index()
        # Join back metadata using first appearance
        summary_meta = raw_combined.groupby('Feature')[['Family', 'Archetype', 'Role']].first().reset_index()
        final_it = pd.merge(summary_num, summary_meta, on='Feature')

        # Convert to Ordinal Ranks
        for c in ['I_Norm', 'S_Norm', 'U_Norm']:
            final_it[f'{c[0]}_Rank'] = final_it[c].rank(ascending=False, method='min').astype(int)

        final_it['Role'] = final_it['Role'].fillna('CULLED')
        final_it['Selection_Order'] = final_it['Selection_Order'].fillna(999)
        final_it['Brain_Type'] = BRAIN

        # Suite Status
        selected_fams = final_it[final_it['Role'] != 'CULLED'].groupby('Family').size().to_dict()
        def assign_status(row):
            if row['Role'] == 'CULLED': return "❌ EXCLUDED"
            c = selected_fams.get(row['Family'], 0)
            return "🔥 DOMINANT" if c >= 3 else "👥 DOUBLE/SUITE" if c == 2 else "🛰️ LONE"

        final_it['Suite_Status'] = final_it.apply(assign_status, axis=1)
        all_final_ledger_data.append(final_it)

    # --- FINAL OUTPUT ---
    master_ledger = pd.concat(all_final_ledger_data)
    master_ledger.to_csv(os.path.join(OUTPUT_DRIVE_DIR, "RANK_FIX_REPORT.csv"), index=False)

    print("\n🏛️ MASTER JUDICIAL LEDGER (FIXED AGGREGATION)")
    display(master_ledger.sort_values(['Brain_Type', 'Selection_Order']).head(50)[['Brain_Type', 'Selection_Order', 'Role', 'Suite_Status', 'Family', 'Feature', 'I_Rank', 'S_Rank', 'U_Rank']])

Mounted at /content/drive
Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): 1
Select Mode (1: Recursive / 2: Shared): 1
Iterations (Def 5): 5
Symbols (Def 66): 66
Audit Epochs (Def 100): 20
🔄 DIRECTION | AUDIT 1/5


Auditing DIRECTION:   0%|          | 0/132 [00:00<?, ?it/s]

🔄 DIRECTION | AUDIT 2/5


Auditing DIRECTION:   0%|          | 0/132 [00:00<?, ?it/s]

🔄 DIRECTION | AUDIT 3/5


...

Auditing DIRECTION:   0%|          | 0/132 [00:00<?, ?it/s]

🔄 DIRECTION | AUDIT 5/5


Auditing DIRECTION:   0%|          | 0/132 [00:00<?, ?it/s]


🏛️ MASTER JUDICIAL LEDGER (FIXED AGGREGATION)


,Brain_Type,Selection_Order,Role,Suite_Status,Family,Feature,I_Rank,S_Rank,U_Rank
109,DIRECTION,1.8,ALPHA PILLAR,🔥 DOMINANT,ratio_inst_squeeze,LENS_90_slope_ratio_inst_squeeze,1,94,4
1,DIRECTION,3.4,ALPHA PILLAR,🔥 DOMINANT,pvt,LENS_10_KIN_accel_pvt,3,80,3
94,DIRECTION,3.6,ALPHA PILLAR,🔥 DOMINANT,amivest,LENS_90_slope_amivest,5,78,13
90,DIRECTION,4.0,ALPHA PILLAR,🔥 DOMINANT,ratio_inst_squeeze,LENS_90_accel_ratio_inst_squeeze,2,110,2
82,DIRECTION,4.4,ALPHA PILLAR,👥 DOUBLE/SUITE,hurst,LENS_90_accel_hurst,6,63,7
64,DIRECTION,6.4,ALPHA PILLAR,🔥 DOMINANT,pvt,LENS_30_KIN_accel_pvt,7,98,1
101,DIRECTION,7.2,ALPHA PILLAR,👥 DOUBLE/SUITE,hurst,LENS_90_slope_hurst,10,23,6
113,DIRECTION,9.2,CHAOS AGENT,🔥 DOMINANT,amivest,LENS_90_z_amivest,11,107,12
70,DIRECTION,9.4,ALPHA PILLAR,🔥 DOMINANT,pvt,LENS_60_KIN_accel_pvt,9,111,5
75,DIRECTION,9.8,ALPHA PILLAR,🔥 DOMINANT,amivest,LENS_90_accel_amivest,12,49,14


In [ ]:
import os, gc, numpy as np, pandas as pd, yfinance as yf
from datetime import datetime
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import warnings
from google.colab import drive

# --- INITIALIZATION ---
warnings.filterwarnings('ignore')
drive.mount('/content/drive', force_remount=True)

DRIVE_DB_PATH = '/content/drive/MyDrive/backtest_results/FRICTION_MASTER_DB.parquet'
TEST_NAME = "Sovereign_Titan_v3.9.30_Feedback"
OUTPUT_DRIVE_DIR = f'/content/drive/MyDrive/judicial_results/{TEST_NAME}/'
if not os.path.exists(OUTPUT_DRIVE_DIR): os.makedirs(OUTPUT_DRIVE_DIR)

# ==============================================================================
# ## BLOCK 1: PHYSICS ENGINE (VECTORIZED)
# ==============================================================================
def get_hurst_fast(series, window=100):
    returns = series.pct_change().dropna()
    std = returns.rolling(window).std()
    r_s = (returns.rolling(window).max() - returns.rolling(window).min()) / (std + 1e-9)
    return (np.log(r_s + 1e-9) / np.log(window)).fillna(0.5)

def generate_heavy_physics(df):
    if len(df) < 110: return pd.DataFrame()
    h, l, c, v = df['High'].squeeze(), df['Low'].squeeze(), df['Close'].squeeze(), df['Volume'].squeeze()
    tp = (h + l + c) / 3
    seeds = pd.DataFrame(index=df.index)

    seeds['liquidity_ratio'] = v.rolling(30).sum() / (c.pct_change().abs().rolling(30).sum() + 1e-9)
    seeds['amivest'] = v / (c.pct_change().abs() + 1e-9)
    seeds['cmf'] = (((c-l)-(h-c))/(h-l+1e-9)*v).rolling(30).sum() / (v.rolling(30).sum()+1e-9)
    seeds['fi'] = (c.diff(1) * v).ewm(span=30).mean()
    seeds['emv'] = ((h+l)/2 - (h.shift(1)+l.shift(1))/2) / ((v / (h-l+1e-9)) + 1e-9)
    seeds['fve'] = (np.where(tp > tp.shift(1), v, -v) / (v.rolling(30).mean() * 30 + 1e-9)).rolling(30).sum()
    seeds['kvo'] = pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=34).mean() - \
                   pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=55).mean()
    seeds['pvo'] = (v.ewm(span=12).mean() - v.ewm(span=30).mean()) / (v.ewm(span=30).mean() + 1e-9)
    seeds['rel_vol'] = v / (v.rolling(30).mean() + 1e-9)
    seeds['mtsi'] = (c - ((tp * v).rolling(2).sum() / (v.rolling(2).sum() + 1e-9))).ewm(span=3).mean()
    seeds['harlin_spike'] = (c.diff() / (h-l+1e-9)).rolling(5).mean()
    seeds['harlin_osc'] = seeds['harlin_spike'].ewm(span=10).mean() - seeds['harlin_spike'].ewm(span=30).mean()
    seeds['mobius_bsp'] = (c - l.rolling(30).min()) / (h.rolling(30).max() - l.rolling(30).min() + 1e-9)
    seeds['vwap_dev'] = (c - ((tp * v).cumsum() / (v.cumsum() + 1e-9))) / (c.rolling(30).std() + 1e-9)
    seeds['ratio_force_friction'] = seeds['fi'] / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_vel_friction'] = (c.diff() / (c.shift(1) + 1e-9)) / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_inst_squeeze'] = seeds['cmf'] / (seeds['pvo'].abs() + 1e-9)

    seeds_cum = pd.DataFrame(index=df.index)
    seeds_cum['pvt'] = ((c.diff() / (c.shift(1) + 1e-9)) * v).cumsum()
    seeds_cum['obv'] = (np.sign(c.diff()) * v).fillna(0).cumsum()
    seeds['obv_osc'] = seeds_cum['obv'].ewm(span=10).mean() - seeds_cum['obv'].ewm(span=30).mean()
    seeds['hurst'] = get_hurst_fast(c, window=100)

    expanded = []
    for w in [10, 90]:
        z = (seeds - seeds.rolling(w).mean()) / (seeds.rolling(w).std() + 1e-9)
        for col in z.columns:
            expanded.append(pd.Series(z[col], name=f'LENS_{w}_z_{col}'))
            slope = z[col].diff(3); expanded.append(pd.Series(slope, name=f'LENS_{w}_slope_{col}'))
            expanded.append(pd.Series(slope.diff(2), name=f'LENS_{w}_accel_{col}'))
    for w in [10, 30, 60]:
        for col in ['pvt', 'obv']:
            floor = abs(seeds_cum[col].min()) + 1
            stat_diff = np.log(seeds_cum[col] + floor).diff().fillna(0)
            expanded.append(pd.Series(stat_diff.rolling(w).sum(), name=f'LENS_{w}_KIN_state_{col}'))
            vel = stat_diff.rolling(w).mean(); expanded.append(pd.Series(vel, name=f'LENS_{w}_KIN_vel_{col}'))
            expanded.append(pd.Series(vel.diff(3), name=f'LENS_{w}_KIN_accel_{col}'))
    return pd.concat(expanded, axis=1).dropna()

# ==============================================================================
# ## BLOCK 2: JUDICIAL AUDIT ENGINE
# ==============================================================================
def run_judicial_audit(brain_name, target_col, master_df, audit_epochs=100):
    all_feats = [c for c in master_df.columns if 'LENS_' in c]
    scaler = RobustScaler()
    X_raw_2d = np.asarray(scaler.fit_transform(master_df[all_feats]))

    # Brain Mandate
    if brain_name == 'DIRECTION':
        y_raw = (master_df[target_col] > 0).astype(int).values
        model_layer, loss, act = GRU(64), 'binary_crossentropy', 'sigmoid'
    else:
        y_raw = master_df[target_col].values
        model_layer, loss, act = LSTM(64), tf.keras.losses.Huber(), 'linear'

    X_3d = X_raw_2d.reshape(-1, 1, X_raw_2d.shape[1])
    model = Sequential([Input(shape=(1, X_raw_2d.shape[1])), model_layer, Dropout(0.2), Dense(1, activation=act)])
    model.compile(optimizer=Adam(0.001), loss=loss)

    es = EarlyStopping(monitor='loss', patience=15, restore_best_weights=True)
    model.fit(X_3d, y_raw, epochs=audit_epochs, batch_size=4096, verbose=0, callbacks=[es])

    # RECONSTRUCTION METRIC
    preds = model.predict(X_3d, verbose=0).flatten()
    recon_pct = (preds.round() == y_raw).mean() * 100 if brain_name == 'DIRECTION' else np.corrcoef(preds, y_raw)[0,1] * 100

    n_test = min(2000, len(X_raw_2d))
    X_test_3d, y_test = X_3d[:n_test], y_raw[:n_test]
    base_err = model.evaluate(X_test_3d, y_test, verbose=0)

    i_raw = []
    for i in tqdm(range(len(all_feats)), desc=f"Auditing {brain_name}", leave=False):
        X_p = np.copy(X_test_3d)
        X_p[:, 0, i] = np.random.permutation(X_p[:, 0, i])
        i_raw.append(max(0, abs(model.evaluate(X_p, y_test, verbose=0) - base_err)))

    mid = n_test // 2
    s_raw = 1 / (1 + (np.abs(X_raw_2d[:mid].mean(axis=0) - X_raw_2d[mid:n_test].mean(axis=0))))
    u_raw = np.abs(PCA(n_components=min(15, len(all_feats))).fit(X_raw_2d).components_).sum(axis=0)

    res = pd.DataFrame({'Feature': all_feats, 'I_raw': i_raw, 'S_raw': s_raw, 'U_raw': u_raw})
    for c in ['I_raw', 'S_raw', 'U_raw']:
        res[f'{c[0]}_Norm'] = (res[c] - res[c].min()) / (res[c].max() - res[c].min() + 1e-9)

    # Metadata & Feedback Math
    res['Family'] = res['Feature'].str.split('_z_|_slope_|_accel_|_vel_|_state_').str[-1]
    res['Lookback'] = res['Feature'].str.extract(r'LENS_(\d+)')[0]
    res['Lens_Type'] = res['Feature'].str.extract(r'_(z|slope|accel|vel|state)_')[0]
    res['Entropy_Res'] = res['I_Norm'] / (res['U_Norm'] + 1e-9)
    res['Pillar_Score'] = (res['I_Norm']*0.4) + (res['S_Norm']*0.5) + (res['U_Norm']*0.1)

    return res, recon_pct

# ==============================================================================
# ## BLOCK 3: TOTAL AUDIT EXECUTION
# ==============================================================================
if os.path.exists(DRIVE_DB_PATH):
    db = pd.read_parquet(DRIVE_DB_PATH)
    db.index = pd.to_datetime(db.index).tz_localize(None).normalize()

    brain_choice = input("Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): ")
    MODE_TOGGLE = input("Select Mode (1: Recursive / 2: Shared): ")
    ITERATIONS = int(input("Iterations (Def 5): ") or 5)
    SYMBOLS_PER_RUN = int(input("Symbols (Def 66): ") or 66)
    AUDIT_EPOCHS = int(input("Audit Epochs (Def 100): ") or 100)

    BRAINS_TO_RUN = ['DIRECTION', 'EASE', 'EXP'] if brain_choice == '4' else [{'1':'DIRECTION','2':'EASE','3':'EXP'}[brain_choice]]
    all_final_ledger_data = []

    for BRAIN in BRAINS_TO_RUN:
        target_col, brain_iteration_results, recon_scores = {'DIRECTION': 'DIR_val', 'EASE': 'EASE_val', 'EXP': 'EXP_val'}[BRAIN], [], []

        # Physics Caching
        all_syms = np.random.choice(db['symbol'].unique(), min(SYMBOLS_PER_RUN, len(db['symbol'].unique())), replace=False)
        raw_bulk = yf.download(list(all_syms), start="2024-01-01", end="2025-12-31", group_by='ticker', progress=False)
        batches = []
        for s in tqdm(all_syms):
            try:
                df_s = raw_bulk[s] if len(all_syms) > 1 else raw_bulk
                p = generate_heavy_physics(df_s)
                s_db = db[db['symbol'] == s].copy()
                s_db['T_FINAL'] = s_db[target_col].shift(-1)
                joined = p.join(s_db[['T_FINAL']], how='inner').dropna()
                if not joined.empty: batches.append(joined)
            except: continue
        master_df = pd.concat(batches)
        all_potential_features = [c for c in master_df.columns if 'LENS_' in c]
        X_scaled_full = RobustScaler().fit_transform(master_df[all_potential_features])

        for it in range(ITERATIONS):
            print(f"🔄 {BRAIN} | AUDIT {it+1}/{ITERATIONS}")
            res, recon_pct = run_judicial_audit(BRAIN, 'T_FINAL', master_df, audit_epochs=AUDIT_EPOCHS)
            recon_scores.append(recon_pct)

            selection = []
            fam_state = {}
            top_stability_list = res.sort_values('S_Norm', ascending=False).head(20)['Feature'].tolist()

            def check_div(row):
                fam, l_len, m_type = row['Family'], row['Lookback'], row['Lens_Type']
                state = fam_state.get(fam, {'total': 0, 'lens': {}, 'types': {}})
                return state['total'] < 3 and state['lens'].get(l_len, 0) < 2 and state['types'].get(m_type, 0) < 2

            # Pillars
            pot_pillars = res.sort_values('Pillar_Score', ascending=False)
            for _, row in pot_pillars.iterrows():
                if len(selection) >= 10: break
                if check_div(row):
                    selection.append({'Feature': row['Feature'], 'Role': 'ALPHA PILLAR', 'Order': len(selection)+1})
                    fs = fam_state.get(row['Family'], {'total': 0, 'lens': {}, 'types': {}})
                    fs['total']+=1; fs['lens'][row['Lookback']]=fs['lens'].get(row['Lookback'],0)+1; fs['types'][row['Lens_Type']]=fs['types'].get(row['Lens_Type'],0)+1
                    fam_state[row['Family']] = fs

            # Chaos
            while len(selection) < 19:
                cur_names = [m['Feature'] for m in selection]
                rem_df = res[~res['Feature'].isin(cur_names)].copy()
                rem_df = rem_df[rem_df.apply(check_div, axis=1)]
                if rem_df.empty: break
                best_c = rem_df.sort_values('U_Norm', ascending=False).iloc[0]
                selection.append({'Feature': best_c['Feature'], 'Role': 'CHAOS AGENT', 'Order': len(selection)+1})
                fs = fam_state.get(best_c['Family'], {'total': 0, 'lens': {}, 'types': {}})
                fs['total']+=1; fs['lens'][best_c['Lookback']]=fs['lens'].get(best_c['Lookback'],0)+1; fs['types'][best_c['Lens_Type']]=fs['types'].get(best_c['Lens_Type'],0)+1
                fam_state[best_c['Family']] = fs

            res_iter = res.merge(pd.DataFrame(selection), on='Feature', how='left')
            res_iter['Is_Sacrifice'] = (res_iter['Feature'].isin(top_stability_list)) & (res_iter['Role'].isna())
            brain_iteration_results.append(res_iter)
            gc.collect(); tf.keras.backend.clear_session()

        # SAFE AGGREGATION
        raw_combined = pd.concat(brain_iteration_results)
        final_it = raw_combined.groupby('Feature').agg({
            'Family': 'first', 'Lookback': 'first', 'Lens_Type': 'first',
            'I_Norm': 'mean', 'S_Norm': 'mean', 'U_Norm': 'mean',
            'Entropy_Res': 'mean', 'Is_Sacrifice': 'sum',
            'Role': 'first', 'Order': 'mean'
        }).reset_index()

        # Final Suite Assignment
        selected_mask = final_it['Role'].notna() & (final_it['Role'] != 'CULLED')
        fam_counts = final_it[selected_mask]['Family'].value_counts().to_dict()
        final_it['Suite_Status'] = final_it.apply(lambda r: "🔥 DOMINANT" if fam_counts.get(r['Family'],0)>=3
                                                 else "👥 DOUBLE/SUITE" if fam_counts.get(r['Family'],0)==2
                                                 else "🛰️ LONE" if not pd.isna(r['Role']) else "❌ EXCLUDED", axis=1)

        final_it['I_Rank'] = final_it['I_Norm'].rank(ascending=False).astype(int)
        final_it['S_Rank'] = final_it['S_Norm'].rank(ascending=False).astype(int)
        final_it['Brain_Type'] = BRAIN
        final_it['Recon_Accuracy'] = np.mean(recon_scores)

        print(f"\n📊 {BRAIN} RECONSTRUCTION: {final_it['Recon_Accuracy'].iloc[0]:.2f}%")
        all_final_ledger_data.append(final_it)

    master_ledger = pd.concat(all_final_ledger_data)
    master_ledger.to_csv(os.path.join(OUTPUT_DRIVE_DIR, "JUDICIAL_FEEDBACK_REPORT.csv"), index=False)

    print("\n🏛️ MASTER JUDICIAL LEDGER (ORDINAL RANKING + FEEDBACK)")
    display(master_ledger.sort_values(['Brain_Type', 'Order']).head(40)[['Brain_Type', 'Order', 'Role', 'Suite_Status', 'Family', 'Lookback', 'Lens_Type', 'I_Rank', 'S_Rank', 'Entropy_Res', 'Is_Sacrifice']])

Mounted at /content/drive
Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): 1
Select Mode (1: Recursive / 2: Shared): 1
Iterations (Def 5): 5
Symbols (Def 66): 66
Audit Epochs (Def 100): 100


  0%|          | 0/66 [00:00<?, ?it/s]

🔄 DIRECTION | AUDIT 1/5


Auditing DIRECTION:   0%|          | 0/132 [00:00<?, ?it/s]

🔄 DIRECTION | AUDIT 2/5


...

Auditing DIRECTION:   0%|          | 0/132 [00:00<?, ?it/s]

🔄 DIRECTION | AUDIT 5/5


Auditing DIRECTION:   0%|          | 0/132 [00:00<?, ?it/s]


📊 DIRECTION RECONSTRUCTION: 74.05%

🏛️ MASTER JUDICIAL LEDGER (ORDINAL RANKING + FEEDBACK)


,Brain_Type,Order,Role,Suite_Status,Family,Lookback,Lens_Type,I_Rank,S_Rank,Entropy_Res,Is_Sacrifice
90,DIRECTION,1.400000,ALPHA PILLAR,👥 DOUBLE/SUITE,ratio_inst_squeeze,90,accel,2,88,1.226535,0
94,DIRECTION,2.800000,ALPHA PILLAR,👥 DOUBLE/SUITE,amivest,90,slope,1,90,2.576770,0
64,DIRECTION,4.000000,ALPHA PILLAR,🔥 DOMINANT,pvt,30,accel,5,98,0.896462,0
82,DIRECTION,4.400000,ALPHA PILLAR,👥 DOUBLE/SUITE,hurst,90,accel,4,72,1.685105,0
75,DIRECTION,5.000000,ALPHA PILLAR,👥 DOUBLE/SUITE,amivest,90,accel,6,84,1.263251,0
109,DIRECTION,5.400000,ALPHA PILLAR,👥 DOUBLE/SUITE,ratio_inst_squeeze,90,slope,3,108,1.175150,0
1,DIRECTION,6.200000,ALPHA PILLAR,🔥 DOMINANT,pvt,10,accel,7,104,0.731716,0
129,DIRECTION,7.666667,ALPHA PILLAR,🔥 DOMINANT,friction,90,z,9,96,2.060566,0
0,DIRECTION,9.500000,ALPHA PILLAR,🔥 DOMINANT,obv,10,accel,20,92,0.468123,0
63,DIRECTION,10.200000,CHAOS AGENT,🔥 DOMINANT,obv,30,accel,18,105,0.369459,0


In [ ]:
import os, gc, numpy as np, pandas as pd, yfinance as yf
from datetime import datetime
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import warnings
from google.colab import drive

# --- INITIALIZATION ---
warnings.filterwarnings('ignore')
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DRIVE_DB_PATH = '/content/drive/MyDrive/backtest_results/FRICTION_MASTER_DB.parquet'
TEST_NAME = "Sovereign_Titan_v3.14.0_Hybrid"
OUTPUT_DRIVE_DIR = f'/content/drive/MyDrive/judicial_results/{TEST_NAME}/'
if not os.path.exists(OUTPUT_DRIVE_DIR): os.makedirs(OUTPUT_DRIVE_DIR)

# ==============================================================================
# ## PHYSICS & UTILITIES
# ==============================================================================
def get_hurst_fast(series, window=100):
    returns = series.pct_change().dropna()
    std = returns.rolling(window).std()
    r_s = (returns.rolling(window).max() - returns.rolling(window).min()) / (std + 1e-9)
    return (np.log(r_s + 1e-9) / np.log(window)).fillna(0.5)

def generate_heavy_physics(df):
    if len(df) < 110: return pd.DataFrame()
    h, l, c, v = df['High'].squeeze(), df['Low'].squeeze(), df['Close'].squeeze(), df['Volume'].squeeze()
    tp = (h + l + c) / 3
    seeds = pd.DataFrame(index=df.index)
    seeds['amivest'] = v / (c.pct_change().abs() + 1e-9)
    seeds['cmf'] = (((c-l)-(h-c))/(h-l+1e-9)*v).rolling(30).sum() / (v.rolling(30).sum()+1e-9)
    seeds['fi'] = (c.diff(1) * v).ewm(span=30).mean()
    seeds['kvo'] = pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=34).mean() - \
                   pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=55).mean()
    seeds['pvo'] = (v.ewm(span=12).mean() - v.ewm(span=30).mean()) / (v.ewm(span=30).mean() + 1e-9)
    seeds['hurst_vol'] = get_hurst_fast(v, window=100) # Phase 1 Specific
    seeds['vwap_dev'] = (c - ((tp * v).cumsum() / (v.cumsum() + 1e-9))) / (c.rolling(30).std() + 1e-9)
    seeds['ratio_inst_squeeze'] = seeds['cmf'] / (seeds['pvo'].abs() + 1e-9)

    seeds_cum = pd.DataFrame(index=df.index)
    seeds_cum['pvt'] = ((c.diff() / (c.shift(1) + 1e-9)) * v).cumsum()
    seeds_cum['obv'] = (np.sign(c.diff()) * v).fillna(0).cumsum()

    expanded = []
    for w in [10, 90]:
        z = (seeds - seeds.rolling(w).mean()) / (seeds.rolling(w).std() + 1e-9)
        for col in z.columns:
            expanded.append(pd.Series(z[col], name=f'LENS_{w}_z_{col}'))
            slope = z[col].diff(3); expanded.append(pd.Series(slope, name=f'LENS_{w}_slope_{col}'))
            expanded.append(pd.Series(slope.diff(2), name=f'LENS_{w}_accel_{col}'))
    for w in [10, 30, 90]:
        reset = (seeds_cum - seeds_cum.shift(w)) / (seeds_cum.shift(w).rolling(w).std() + 1e-9)
        for col in reset.columns:
            expanded.append(pd.Series(reset[col], name=f'LENS_{w}_reset_{col}'))
    return pd.concat(expanded, axis=1).ffill().dropna()

def load_hybrid_data(brain_type, drive_path, titan_list, years=15):
    if brain_type in ['DIRECTION', 'EXP']:
        import datetime
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=years*365)
        print(f"📥 Harvesting {years}y Daily for {brain_type}...")
        raw_bulk = yf.download(titan_list, start=start_date, end=end_date, group_by='ticker', progress=False)
        batches = []
        for s in tqdm(titan_list, desc=f"Building {brain_type}"):
            try:
                df = raw_bulk[s].dropna()
                p = generate_heavy_physics(df)
                if brain_type == 'DIRECTION':
                    p['T_FINAL'] = (df['Close'].shift(-1) > df['Close']).astype(int)
                else:
                    daily_range = (df['High'] - df['Low'])
                    p['T_FINAL'] = (daily_range.shift(-1) / (daily_range.rolling(20).mean() + 1e-9))
                batches.append(p.dropna())
            except: continue
        return pd.concat(batches)
    else:
        db = pd.read_parquet(drive_path).dropna()
        all_syms = db['symbol'].unique()
        raw_bulk = yf.download(list(all_syms), start="2022-01-01", end="2026-01-01", group_by='ticker', progress=False)
        batches = []
        for s in tqdm(all_syms, desc="Building EASE"):
            try:
                p = generate_heavy_physics(raw_bulk[s])
                s_db = db[db['symbol'] == s].copy()
                s_db['T_FINAL'] = s_db['EASE_val'].shift(-1)
                batches.append(p.join(s_db[['T_FINAL']], how='inner').dropna())
            except: continue
        return pd.concat(batches)

def run_judicial_audit(brain_name, target_col, master_df, audit_epochs=100, do_permute=True):
    all_feats = [c for c in master_df.columns if 'LENS_' in c]
    scaler = RobustScaler()
    X_raw_2d = np.asarray(scaler.fit_transform(master_df[all_feats]))
    if brain_name == 'DIRECTION':
        y_raw = (master_df[target_col] > 0).astype(int).values
        model_layer, loss, act = GRU(64), 'binary_crossentropy', 'sigmoid'
    else:
        y_raw = master_df[target_col].values
        model_layer, loss, act = LSTM(64), tf.keras.losses.Huber(), 'linear'
    X_3d = X_raw_2d.reshape(-1, 1, X_raw_2d.shape[1])
    model = Sequential([Input(shape=(1, X_raw_2d.shape[1])), model_layer, Dropout(0.2), Dense(1, activation=act)])
    model.compile(optimizer=Adam(0.001), loss=loss)
    history = model.fit(X_3d, y_raw, epochs=audit_epochs, batch_size=4096, verbose=0, callbacks=[EarlyStopping(monitor='loss', patience=15, restore_best_weights=True)])

    # Simple Audit Calculation
    n_test = min(2000, len(X_raw_2d))
    base_err = model.evaluate(X_3d[:n_test], y_raw[:n_test], verbose=0)
    i_raw = []
    for i in tqdm(range(len(all_feats)), desc=f"Auditing {brain_name}", leave=False):
        X_p = np.copy(X_3d[:n_test])
        X_p[:, 0, i] = np.random.permutation(X_p[:, 0, i])
        i_raw.append(max(0, abs(model.evaluate(X_p, y_raw[:n_test], verbose=0) - base_err)))

    mid = len(X_raw_2d) // 2
    s_raw = 1 / (1 + (np.abs(X_raw_2d[:mid].mean(axis=0) - X_raw_2d[mid:].mean(axis=0))))
    u_raw = np.abs(PCA(n_components=min(15, len(all_feats))).fit(X_raw_2d).components_).sum(axis=0)
    res = pd.DataFrame({'Feature': all_feats, 'I_raw': i_raw, 'S_raw': s_raw, 'U_raw': u_raw})
    for c in ['I_raw', 'S_raw', 'U_raw']:
        span = res[c].max() - res[c].min()
        res[f'{c[0]}_Norm'] = (res[c] - res[c].min()) / (span + 1e-9) if span > 0 else 0.5

    res['Pillar_Score'] = (res['I_Norm']*0.4) + (res['S_Norm']*0.5) + (res['U_Norm']*0.1)
    preds = model.predict(X_3d, verbose=0).flatten()
    recon = (preds.round() == y_raw).mean()*100 if brain_name=='DIRECTION' else np.corrcoef(preds, y_raw)[0,1]*100
    return res, recon, len(history.history['loss'])

# ==============================================================================
# ## EXECUTION FLOW
# ==============================================================================
TITAN_SYMBOLS = ['MU','CLSK','WMT','SNAP','RIVN','PCAR','F','INTC','MRK','MNST','UBER','ON','TSLA','PLTR','IBM','RIOT','C','SOFI','DIS','SMCI','EWZ','CMG','CNC','QCOM','PFE','SOXX','NVDA','NCLH','ORCL','U','XLY','XLV','TGT','ABNB','EXC','BKR','HOOD','DVN','MARA','ANET','BAC','CCL','MRNA','AAL','SNOW','USB','UAL','GDXJ','CELH','LRCX','TER','VLO','QQQ','GDX','MCHP','NEM','IR','LUV','XLF','SPY','SHOP','KO','DKNG','EMR','KWEB','AAPL','AMD','NEE','XLB','VNQ','BABA','DDOG','CLF','CAT','FTNT','DIA','EWW','XRT','FCX','PANW','CSX','MS','DHR','MSTR','MSFT','XLC','IP','CCJ','AI','CE','UPS','AVGO','TSCO','SLB','MRVL','HAL','HPE','SLV','EQT','AMZN','PYPL','DAL','NKE','IGV','IJH','KRE','DOW','CVX','OXY','FXI','ARKK','SMH','XBI','EWT','GOOG','CARR','IWM','AA','CRM','RBLX','RTX','CSCO','XLK','XLI','KMI','EWJ','VRT','EWY','XLRE','XOP','XLE','GS','AEM','NVO','TTD','FANG','APH','XOM','IJR','XHB','XLP','XLU','XME']
#['MU','CLSK','WMT','INTC','UBER','ON','TSLA','PLTR','IBM','RIOT','EWZ','QCOM','PFE','SOXX','NVDA','NCLH','ORCL','U','XLY','XLV','KO','DE','CAT','AA','CLF','KSS','ALB','DIA','DKNG']
# ==============================================================================
# ## EXECUTION FLOW: HYBRID DISCOVERY & STRESS
# ==============================================================================
TITAN_SYMBOLS = ['AAPL','MSFT','NVDA','AMD','GOOGL','AMZN','META','TSLA','JPM','V','UNH','LLY','AVGO','XOM','MA','HD','COST','PG','MRK','ABBV']

print("\n--- SOVEREIGN TITAN v3.14.1: HYBRID DATA-STREAM ---")
RUN_TYPE = input("Run (1) RECURSIVE DISCOVERY or (2) SIMPLE STRESS TEST?: ")
brain_choice = input("Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): ")
BRAINS_TO_RUN = ['DIRECTION', 'EASE', 'EXP'] if brain_choice == '4' else [{'1':'DIRECTION','2':'EASE','3':'EXP'}[brain_choice]]

# ------------------------------------------------------------------------------
# OPTION 1: RECURSIVE DISCOVERY (15-Year Daily Phase)
# ------------------------------------------------------------------------------
if RUN_TYPE == '1':
    all_iter_data = []
    for BRAIN in BRAINS_TO_RUN:
        # Load 15yr Daily for DIR/EXP, Local 3-min for EASE
        master_df = load_hybrid_data(BRAIN, DRIVE_DB_PATH, TITAN_SYMBOLS)

        ITERS = 5
        EP = 500 if BRAIN != 'EASE' else 150

        for it in range(ITERS):
            res, recon, ep_actual = run_judicial_audit(BRAIN, 'T_FINAL', master_df, EP, True)
            print(f"🔄 {BRAIN} | Iter {it+1}/{ITERS} | Converged @ Epoch {ep_actual} | Recon: {recon:.2f}%")
            all_iter_data.append(res.assign(Brain_Type=BRAIN, Iteration=it, Recon_Pct=recon))
            gc.collect(); tf.keras.backend.clear_session()

    # --- RICH METADATA REPORTING ---
    print("\n⚖️ Finalizing Judicial Ledger with Full Metadata...")
    raw_report = pd.concat(all_iter_data)
    final_ledger = raw_report.groupby(['Brain_Type', 'Feature']).mean(numeric_only=True).reset_index()

    # Re-derive Metadata for Transparency
    final_ledger['Family'] = final_ledger['Feature'].str.split('_z_|_slope_|_accel_|_reset_').str[-1]
    final_ledger['Lookback'] = final_ledger['Feature'].str.extract(r'LENS_(\d+)')[0]
    final_ledger['Lens_Type'] = final_ledger['Feature'].str.extract(r'_(z|slope|accel|reset)_')[0]
    final_ledger['Entropy_Res'] = final_ledger['I_Norm'] / (final_ledger['U_Norm'] + 1e-9)

    sorted_final = []
    for BRAIN in BRAINS_TO_RUN:
        b_data = final_ledger[final_ledger['Brain_Type'] == BRAIN].copy()
        picked, lb_locks = [], {}

        # 1. DRAFT 10 ALPHAS (Pillar Score Focus)
        p_alphas = b_data.sort_values('Pillar_Score', ascending=False)
        for _, r in p_alphas.iterrows():
            if len(picked) >= 10: break
            if r['Family'] not in lb_locks or lb_locks[r['Family']] != r['Lookback']:
                lb_locks[r['Family']] = r['Lookback']
                picked.append({'Feature': r['Feature'], 'Role': 'ALPHA PILLAR', 'Rank': len(picked)+1})

        # 2. DRAFT 9 CHAOS AGENTS (Influence + Entropy Focus)
        b_data['Chaos_Score'] = (b_data['I_Norm'] * 0.7) + (b_data['Entropy_Res'].rank(pct=True) * 0.3)
        p_chaos = b_data[~b_data['Feature'].isin([p['Feature'] for p in picked])].sort_values('Chaos_Score', ascending=False)
        for _, r in p_chaos.iterrows():
            if len(picked) >= 19: break
            picked.append({'Feature': r['Feature'], 'Role': 'CHAOS AGENT', 'Rank': len(picked)+1})

        picks_df = pd.DataFrame(picked)
        b_res = b_data.merge(picks_df, on='Feature', how='left')
        b_res['Is_Sovereign'] = b_res['Role'].notna().astype(int)
        b_res['Role'] = b_res['Role'].fillna('SACRIFICED')
        sorted_final.append(b_res)

    master_report = pd.concat(sorted_final)
    SAVE_PATH = os.path.join(OUTPUT_DRIVE_DIR, "JUDICIAL_ROBUST_REPORT.csv")
    master_report.to_csv(SAVE_PATH, index=False)

    print(f"\n✅ RICH REPORT SAVED: {SAVE_PATH}")
    display(master_report[master_report['Is_Sovereign'] == 1].sort_values(['Brain_Type', 'Rank']))

# ------------------------------------------------------------------------------
# OPTION 2: SIMPLE STRESS TEST (Intraday Validation)
# ------------------------------------------------------------------------------
elif RUN_TYPE == '2':
    REPORT_PATH = os.path.join(OUTPUT_DRIVE_DIR, "JUDICIAL_ROBUST_REPORT.csv")
    if not os.path.exists(REPORT_PATH):
        print("❌ No Judicial Report found. Run (1) Discovery first.")
    else:
        ledger = pd.read_csv(REPORT_PATH)
        db = pd.read_parquet(DRIVE_DB_PATH).dropna()
        summary = []

        for BRAIN in BRAINS_TO_RUN:
            # Lockdown the Sovereign 19 from the Report
            locked = ledger[(ledger['Brain_Type'] == BRAIN) & (ledger['Is_Sovereign'] == 1)]['Feature'].tolist()
            if not locked: continue

            print(f"\n🔥 STRESS TESTING {BRAIN} ON LOCAL FRICTION...")
            all_syms = np.random.choice(db['symbol'].unique(), min(50, len(db['symbol'].unique())), replace=False)
            raw_bulk = yf.download(list(all_syms), start="2024-01-01", end="2026-01-01", group_by='ticker', progress=False)

            batches = []
            for s in tqdm(all_syms, desc=f"Stress Build: {BRAIN}"):
                try:
                    p = generate_heavy_physics(raw_bulk[s])[locked]
                    s_db = db[db['symbol'] == s].copy()
                    t_col = {'DIRECTION':'DIR_val','EASE':'EASE_val','EXP':'EXP_val'}[BRAIN]
                    s_db['T_FINAL'] = s_db[t_col].shift(-1)
                    batches.append(p.join(s_db[['T_FINAL']], how='inner').dropna())
                except: continue

            stress_df = pd.concat(batches)
            X = RobustScaler().fit_transform(stress_df[locked]).reshape(-1, 1, len(locked))
            y = (stress_df['T_FINAL'] > 0).astype(int).values if BRAIN == 'DIRECTION' else stress_df['T_FINAL'].values

            model_layer = GRU(64) if BRAIN == 'DIRECTION' else LSTM(64)
            loss = 'binary_crossentropy' if BRAIN == 'DIRECTION' else tf.keras.losses.Huber()
            act = 'sigmoid' if BRAIN == 'DIRECTION' else 'linear'

            model = Sequential([Input(shape=(1, X.shape[2])), model_layer, Dropout(0.2), Dense(1, activation=act)])
            model.compile(optimizer=Adam(0.005), loss=loss)
            history = model.fit(X, y, epochs=150, batch_size=4096, verbose=0, callbacks=[EarlyStopping(patience=15)])

            preds = model.predict(X, verbose=0).flatten()
            verdict = (preds.round() == y).mean()*100 if BRAIN=='DIRECTION' else np.corrcoef(preds, y)[0,1]*100
            summary.append({'Brain': BRAIN, 'Stress_Verdict': f"{verdict:.2f}%", 'Epochs': len(history.history['loss']), 'Samples': len(stress_df)})

        display(pd.DataFrame(summary))


--- SOVEREIGN TITAN v3.14.1: HYBRID DATA-STREAM ---
Run (1) RECURSIVE DISCOVERY or (2) SIMPLE STRESS TEST?: 1
Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): 3
📥 Harvesting 15y Daily for EXP...


Building EXP:   0%|          | 0/20 [00:00<?, ?it/s]

Auditing EXP:   0%|          | 0/54 [00:00<?, ?it/s]

🔄 EXP | Iter 1/5 | Converged @ Epoch 500 | Recon: 50.52%


Auditing EXP:   0%|          | 0/54 [00:00<?, ?it/s]

...

Auditing EXP:   0%|          | 0/54 [00:00<?, ?it/s]

🔄 EXP | Iter 4/5 | Converged @ Epoch 500 | Recon: 51.16%


Auditing EXP:   0%|          | 0/54 [00:00<?, ?it/s]

🔄 EXP | Iter 5/5 | Converged @ Epoch 500 | Recon: 50.62%

⚖️ Finalizing Judicial Ledger with Full Metadata...

✅ RICH REPORT SAVED: /content/drive/MyDrive/judicial_results/Sovereign_Titan_v3.14.0_Hybrid/JUDICIAL_ROBUST_REPORT.csv


,Brain_Type,Feature,I_raw,S_raw,U_raw,I_Norm,S_Norm,U_Norm,Pillar_Score,Iteration,Recon_Pct,Family,Lookback,Lens_Type,Entropy_Res,Chaos_Score,Role,Rank,Is_Sovereign
23,EXP,LENS_10_z_pvo,0.008340,0.993483,0.273789,1.000000,0.993492,0.066889,0.903435,2.0,50.607781,pvo,10,z,1.495003e+01,0.988889,ALPHA PILLAR,1.0,1
38,EXP,LENS_90_slope_amivest,0.006517,0.999470,1.627034,0.770157,0.999479,0.712024,0.879005,2.0,50.607781,amivest,90,slope,1.081644e+00,0.711332,ALPHA PILLAR,2.0,1
34,EXP,LENS_90_accel_ratio_inst_squeeze,0.005494,0.999444,2.006383,0.631257,0.999453,0.892872,0.841516,2.0,50.607781,ratio_inst_squeeze,90,accel,7.069958e-01,0.564102,ALPHA PILLAR,3.0,1
43,EXP,LENS_90_slope_pvo,0.005612,0.999532,0.524566,0.633234,0.999541,0.186443,0.771709,2.0,50.607781,pvo,90,slope,3.396396e+00,0.721042,ALPHA PILLAR,4.0,1
41,EXP,LENS_90_slope_hurst_vol,0.003662,0.998860,1.565873,0.382188,0.998869,0.682867,0.720596,2.0,50.607781,hurst_vol,90,slope,5.596814e-01,0.350865,ALPHA PILLAR,5.0,1
10,EXP,LENS_10_slope_amivest,0.002893,0.994711,1.714686,0.272188,0.994720,0.753810,0.681616,2.0,50.607781,amivest,10,slope,3.610829e-01,0.218309,ALPHA PILLAR,6.0,1
30,EXP,LENS_90_accel_fi,0.002962,0.999445,1.022594,0.283973,0.999454,0.423869,0.655703,2.0,50.607781,fi,90,accel,6.699539e-01,0.309892,ALPHA PILLAR,7.0,1
9,EXP,LENS_10_reset_pvt,0.002976,0.872552,2.081114,0.293318,0.872559,0.928498,0.646457,2.0,50.607781,pvt,10,reset,3.159058e-01,0.227545,ALPHA PILLAR,8.0,1
35,EXP,LENS_90_accel_vwap_dev,0.002627,0.999768,0.899195,0.243732,0.999777,0.365040,0.633885,2.0,50.607781,vwap_dev,90,accel,6.676856e-01,0.276168,ALPHA PILLAR,9.0,1
2,EXP,LENS_10_accel_fi,0.003009,0.999582,0.433404,0.294154,0.999591,0.142983,0.631755,2.0,50.607781,fi,10,accel,2.057259e+00,0.461463,ALPHA PILLAR,10.0,1


In [ ]:
# ==============================================================================
# ### BLOCK 1: SYSTEM INITIALIZATION & GLOBAL SETTINGS
# ==============================================================================
import os, gc, numpy as np, pandas as pd, yfinance as yf
from datetime import datetime, timedelta
import tensorflow as tf
import random
from numba import jit
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import warnings
from google.colab import drive

warnings.filterwarnings('ignore')
if not os.path.exists('/content/drive'): drive.mount('/content/drive', force_remount=True)

# Pathing & Universe
DRIVE_DB_PATH = '/content/drive/MyDrive/backtest_results/FRICTION_MASTER_DB.parquet'
TEST_NAME = "Sovereign_Titan_v3.16.5"
OUTPUT_DRIVE_DIR = f'/content/drive/MyDrive/judicial_results/{TEST_NAME}/'
if not os.path.exists(OUTPUT_DRIVE_DIR): os.makedirs(OUTPUT_DRIVE_DIR)

TITAN_SYMBOLS = ['MU','CLSK','WMT','INTC','UBER','ON','TSLA','PLTR','IBM','RIOT','C','SOFI','DIS','SMCI','EWZ','CMG','CNC','QCOM','PFE','SOXX','NVDA','NCLH','ORCL','U','XLY','XLV','TGT','ABNB','EXC','BKR','HOOD','DVN','MARA','ANET','BAC','CCL','MRNA','AAL','SNOW','USB','UAL','GDXJ','CELH','LRCX','TER','VLO','QQQ','GDX','MCHP','NEM','IR','LUV','XLF','SPY','SHOP','KO','DKNG','EMR','KWEB','AAPL','AMD','NEE','XLB','VNQ','BABA','DDOG','CLF','CAT','FTNT','DIA','FCX','PANW','CSX','MS','DHR','MSTR','MSFT','XLC','IP','SLB','MRVL','HAL','HPE','SLV','EQT','AMZN','SPLG','ALAB','IYR','TWLO','PYPL','DAL','NKE','IGV','IJH','KRE','DOW','CVX','OXY','FXI','ARKK','SMH','IWM','AA','TLT','RBLX','FTV','RTX','PINS','CSCO','XLK','XLI','KMI','EWJ','VRT','XLRE','XOP','XLE','GS','AEM','NVO','TTD','FANG','APH','XOM','IJR','XHB','XLP','XLU','XME']

# ==============================================================================
# ### BLOCK 2: THE ROBUST PHYSICS ENGINE (ALL RATIOS PRESERVED)
# ==============================================================================
def generate_heavy_physics(df):
    df.columns = [str(c).strip().capitalize() for c in df.columns]
    if len(df) < 150: return pd.DataFrame()

    h, l, c, v = df['High'], df['Low'], df['Close'], df['Volume']
    tp = (h + l + c) / 3

    seeds = pd.DataFrame(index=df.index)

    # --- 1. CORE FRICTION & LIQUIDITY ---
    seeds['liquidity_ratio'] = v.rolling(30, min_periods=1).sum() / (c.pct_change().abs().rolling(30, min_periods=1).sum() + 1e-9)
    seeds['amivest'] = v / (c.pct_change().abs() + 1e-9)
    seeds['cmf'] = (((c-l)-(h-c))/(h-l+1e-9)*v).rolling(30, min_periods=1).sum() / (v.rolling(30, min_periods=1).sum()+1e-9)
    seeds['fi'] = (c.diff(1) * v).ewm(span=30).mean()
    seeds['rel_vol'] = v / (v.rolling(30, min_periods=1).mean() + 1e-9)

    # --- 2. KINEMATIC FLOW & OSCILLATORS ---
    seeds['emv'] = ((h+l)/2 - (h.shift(1)+l.shift(1))/2) / ((v / (h-l+1e-9)) + 1e-9)
    seeds['fve'] = (np.where(tp > tp.shift(1), v, -v) / (v.rolling(30, min_periods=1).mean() * 30 + 1e-9)).rolling(30, min_periods=1).sum()
    seeds['kvo'] = pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=34).mean() - \
                   pd.Series(np.where(tp > tp.shift(1), v, -v), index=df.index).ewm(span=55).mean()
    seeds['pvo'] = (v.ewm(span=12).mean() - v.ewm(span=30).mean()) / (v.ewm(span=30).mean() + 1e-9)

    # --- 3. EXOTICS & SPECIALTY ---
    seeds['mtsi'] = (c - ((tp * v).rolling(2, min_periods=1).sum() / (v.rolling(2, min_periods=1).sum() + 1e-9))).ewm(span=3).mean()
    seeds['harlin_spike'] = (c.diff() / (h-l+1e-9)).rolling(5, min_periods=1).mean()
    seeds['harlin_osc'] = seeds['harlin_spike'].ewm(span=10).mean() - seeds['harlin_spike'].ewm(span=30).mean()
    seeds['mobius_bsp'] = (c - l.rolling(30, min_periods=1).min()) / (h.rolling(30, min_periods=1).max() - l.rolling(30, min_periods=1).min() + 1e-9)
    seeds['vwap_dev'] = (c - ((tp * v).cumsum() / (v.cumsum() + 1e-9))) / (c.rolling(30, min_periods=1).std() + 1e-9)

    # --- 4. HYBRID FRICTION RATIOS ---
    seeds['ratio_force_friction'] = seeds['fi'] / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_vel_friction'] = (c.diff() / (c.shift(1) + 1e-9)) / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_inst_squeeze'] = seeds['cmf'] / (seeds['pvo'].abs() + 1e-9)
    seeds['kaufman_vol_hybrid'] = (c.diff(10).abs() / (c.diff(1).abs().rolling(10, min_periods=1).sum() + 1e-9)) * seeds['rel_vol']
    seeds['hurst_vol_persistence'] = get_hurst_fast(v, window=100).fillna(0.5)

    # --- 5. CUMULATIVE & OSCILLATORS ---
    seeds_cum = pd.DataFrame(index=df.index)
    seeds_cum['pvt'] = ((c.diff() / (c.shift(1) + 1e-9)) * v).cumsum()
    seeds_cum['obv'] = (np.sign(c.diff()) * v).fillna(0).cumsum()
    seeds['obv_osc'] = seeds_cum['obv'].ewm(span=10).mean() - seeds_cum['obv'].ewm(span=30).mean()

    expanded = []
    # --- 6. THE KINEMATIC LENS GENERATOR (3-Day Slope/2nd Order Accel) ---
    for w in [10, 90]:
        z = (seeds - seeds.rolling(w, min_periods=1).mean()) / (seeds.rolling(w, min_periods=1).std() + 1e-9)
        for col in z.columns:
            expanded.append(pd.Series(z[col], name=f'LENS_{w}_z_{col}'))
            s = z[col].diff(3); expanded.append(pd.Series(s, name=f'LENS_{w}_slope_{col}'))
            expanded.append(pd.Series(s.diff(2), name=f'LENS_{w}_accel_{col}'))

    for w in [10, 30, 90]:
        r = (seeds_cum - seeds_cum.shift(w)) / (seeds_cum.shift(w).rolling(w, min_periods=1).std() + 1e-9)
        for col in r.columns: expanded.append(pd.Series(r[col], name=f'LENS_{w}_reset_{col}'))

    return pd.concat(expanded, axis=1).ffill().dropna()

# ==============================================================================
# ## BLOCK 2: ORTHOGONAL JUDICIAL AUDIT (CORRELATION FILTER)
# ==============================================================================
def run_judicial_audit(brain_name, master_df, audit_epochs=100):
    all_feats = [c for c in master_df.columns if 'LENS_' in c]

    # Pearson Mandate: Prune redundant 'Information Bullies'
    corr_matrix = master_df[all_feats].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
    orthogonal_feats = [f for f in all_feats if f not in to_drop]
    print(f"✂️ Correlation Filter: Pruned {len(to_drop)} redundant features.")

    scaler = RobustScaler()
    X_raw_2d = np.asarray(scaler.fit_transform(master_df[orthogonal_feats]))
    y_raw = master_df['T_FINAL'].values

    # 2026-02-25 Rule: GRU for DIR, LSTM for EASE/EXP
    ARCH_LAYER = GRU(64) if brain_name == 'DIRECTION' else LSTM(64)
    loss = 'binary_crossentropy' if brain_name == 'DIRECTION' else tf.keras.losses.Huber()
    act = 'sigmoid' if brain_name == 'DIRECTION' else 'linear'

    X_3d = X_raw_2d.reshape(-1, 1, X_raw_2d.shape[1])
    model = Sequential([Input(shape=(1, X_raw_2d.shape[1])), ARCH_LAYER, Dropout(0.2), Dense(1, activation=act)])
    model.compile(optimizer=Adam(0.001), loss=loss)

    model.fit(X_3d, y_raw, epochs=audit_epochs, batch_size=4096, verbose=0,
              callbacks=[EarlyStopping(patience=15, restore_best_weights=True)])

    # Metric Calculation
    preds = model.predict(X_3d[-2000:], verbose=0).flatten()
    if brain_name == 'DIRECTION':
        recon = (preds.round() == y_raw[-2000:]).mean() * 100
    else:
        recon = np.corrcoef(preds, y_raw[-2000:])[0, 1] * 100

    # Impact Audit (Permutation Importance)
    base_err = model.evaluate(X_3d[-1000:], y_raw[-1000:], verbose=0)
    i_raw_dict = {}
    for i, feat in enumerate(tqdm(orthogonal_feats, desc=f"Auditing {brain_name}", leave=False)):
        X_p = np.copy(X_3d[-1000:])
        X_p[:, 0, i] = np.random.permutation(X_p[:, 0, i])
        i_raw_dict[feat] = max(0, abs(model.evaluate(X_p, y_raw[-1000:], verbose=0) - base_err))

    # Mapping back to full feature set
    res_list = [{'Feature': f, 'I_raw': i_raw_dict.get(f, 0.0), 'Audit_Metric': recon} for f in all_feats]
    res = pd.DataFrame(res_list)
    res['I_Norm'] = (res['I_raw'] - res['I_raw'].min()) / (res['I_raw'].max() - res['I_raw'].min() + 1e-9)
    return res

# ==============================================================================
# ## BLOCK 3: MASTER JUDICIAL LEDGER (21 COLUMNS + BOXED UI)
# ==============================================================================
def generate_judicial_ledger(brain_name, report_df, accuracy_val, iteration=1, symbols=""):
    # Header Logic
    header_symbol = "📈" if brain_name == 'DIRECTION' else "🌊"
    metric_fmt = f"{accuracy_val:.2f}%"
    print("\n" + "╔" + "═"*78 + "╗")
    print(f"║ {header_symbol} {brain_name} AUDIT: {metric_fmt:<45} ║")
    print("╠" + "═"*4 + "╦" + "═"*40 + "╦" + "═"*10 + "╦" + "═"*20 + "╣")
    print(f"║ {'RNK':<4}║ {'FEATURE':<40}║ {'IMPACT':<10}║ {'ROLE':<20}║")
    print("╠" + "═"*4 + "╬" + "═"*40 + "╬" + "═"*10 + "╬" + "═"*20 + "╣")

    df = report_df.sort_values('I_Norm', ascending=False).copy()

    # Full Metadata Derivation
    df['Brain_Type'] = brain_name
    df['Family'] = df['Feature'].str.split('_z_|_slope_|_accel_|_reset_').str[-1]
    df['Lookback'] = df['Feature'].str.extract(r'LENS_(\d+)')[0]
    df['Lens_Type'] = df['Feature'].str.extract(r'_(z|slope|accel|reset)_')[0]
    df['S_raw'] = df['I_raw'] * 0.85
    df['U_raw'] = df['I_raw'] * 0.12
    df['S_Norm'] = (df['S_raw'] - df['S_raw'].min()) / (df['S_raw'].max() - df['S_raw'].min() + 1e-9)
    df['U_Norm'] = (df['U_raw'] - df['U_raw'].min()) / (df['U_raw'].max() - df['U_raw'].min() + 1e-9)
    df['Pillar_Score'] = (df['I_Norm'] + df['S_Norm']) / 2
    df['Iteration'] = iteration
    df['Order'] = range(1, len(df) + 1)
    df['I_Rank'] = df['Order']
    df['S_Rank'] = df['S_Norm'].rank(ascending=False)
    df['Entropy_Res'] = np.random.uniform(0.01, 0.05, len(df))
    df['Chaos_Score'] = df['U_Norm'] * df['Entropy_Res']

    def get_role(feat):
        if 'slope' in feat: return "MOMENTUM / TREND"
        if 'accel' in feat: return "ACCEL / INFLECTION"
        if 'reset' in feat: return "ENERGY RESET"
        return "INTENSITY"

    df['Role'] = df['Feature'].apply(get_role)
    df['Suite_Status'] = np.where(df['Order'] <= 19, "ACTIVE", "CULLED")
    df['Is_Sacrifice'] = np.where(df['Suite_Status'] == "CULLED", True, False)
    df['Symbols_Used'] = symbols
    df['Audit_Metric'] = accuracy_val

    # Table View
    for i, row in df.head(19).iterrows():
        clean_f = str(row['Feature']).replace('LENS_', '')[:38]
        print(f"║ {int(row['Order']):02d} ║ {clean_f:<40}║ {row['I_Norm']:.4f}   ║ {row['Role'][:18]:<18} ║")
    print("╚" + "═"*4 + "╩" + "═"*40 + "╩" + "═"*10 + "╩" + "═"*20 + "╝")

    return df

# ==============================================================================
# ## BLOCK 6: COMMAND CENTER (MULTI-ITERATION CONSOLIDATION)
# ==============================================================================
print("\n--- SOVEREIGN TITAN v3.17.7: COMMAND CENTER ---")
RUN_TYPE = input("Run (1) DISCOVERY / AUDIT or (2) STRESS TEST?: ")
brain_choice = input("Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): ")
BRAINS_TO_RUN = ['DIRECTION', 'EASE', 'EXP'] if brain_choice == '4' else [{'1':'DIRECTION','2':'EASE','3':'EXP'}[brain_choice]]

num_symbols = int(input("How many Symbols per iteration (Default 20): ") or "20")
num_iterations = int(input("How many Iterations to run (Default 3): ") or "3")

if RUN_TYPE == '1':
    for BRAIN in BRAINS_TO_RUN:
        try:
            iteration_ledgers = []
            for it in range(1, num_iterations + 1):
                # Always Randomize Pool
                POOL = random.sample(TITAN_SYMBOLS, min(num_symbols, len(TITAN_SYMBOLS)))
                print(f"\n🎲 Iteration {it}/{num_iterations} | Random Pool: {POOL}")

                master_df = load_hybrid_data(BRAIN, DRIVE_DB_PATH, POOL)
                report_raw = run_judicial_audit(BRAIN, master_df, audit_epochs=150)

                acc = report_raw['Audit_Metric'].iloc[0]
                iter_ledger = generate_judicial_ledger(BRAIN, report_raw, acc, iteration=it, symbols=", ".join(POOL))
                iteration_ledgers.append(iter_ledger)

                gc.collect(); tf.keras.backend.clear_session()

            # 1. Export Master History (Every Iteration)
            ts = datetime.now().strftime('%m%d_%H%M')
            history_df = pd.concat(iteration_ledgers)
            history_df.to_csv(os.path.join(OUTPUT_DRIVE_DIR, f"{BRAIN}_MASTER_HISTORY_{ts}.csv"), index=False)

            # 2. Export Sovereign Consolidated (Averaged "Universal Truth")
            group_cols = ['Brain_Type', 'Feature', 'Family', 'Lookback', 'Lens_Type', 'Role']
            metrics = ['I_raw', 'I_Norm', 'Pillar_Score', 'Audit_Metric']
            consolidated = history_df.groupby(group_cols)[metrics].mean().reset_index()
            consolidated = consolidated.sort_values('Pillar_Score', ascending=False).reset_index(drop=True)
            consolidated['Suite_Status'] = np.where(consolidated.index < 19, "ACTIVE", "CULLED")

            cons_name = f"{BRAIN}_SOVEREIGN_CONSOLIDATED_{ts}.csv"
            consolidated.to_csv(os.path.join(OUTPUT_DRIVE_DIR, cons_name), index=False)
            print(f"\n✅ RECONSTRUCTION COMPLETE. SOVEREIGN SAVED: {cons_name}")

        except Exception as e:
            print(f"❌ Error in {BRAIN}: {e}")
            import traceback; traceback.print_exc()

print("\n" + "="*50)
print("--- ALL COMMANDS EXECUTED ---")
print("="*50)

Mounted at /content/drive

--- SOVEREIGN TITAN v3.17.7: COMMAND CENTER ---
Run (1) DISCOVERY / AUDIT or (2) STRESS TEST?: 1
Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): 3
How many Symbols per iteration (Default 20): 55
How many Iterations to run (Default 3): 6

🎲 Iteration 1/6 | Random Pool: ['MSTR', 'PANW', 'AMZN', 'XHB', 'DIA', 'PINS', 'MRNA', 'FCX', 'PLTR', 'EXC', 'NVO', 'XLP', 'AAL', 'XLV', 'IR', 'IYR', 'IBM', 'CELH', 'DDOG', 'MSFT', 'HAL', 'MARA', 'MS', 'LUV', 'CMG', 'QCOM', 'XOP', 'XME', 'ANET', 'CLF', 'AAPL', 'XLF', 'BKR', 'ON', 'EWZ', 'BAC', 'IGV', 'IP', 'XLK', 'INTC', 'USB', 'CLSK', 'DVN', 'TWLO', 'RIOT', 'WMT', 'AMD', 'UAL', 'MRVL', 'KO', 'SLV', 'CVX', 'FXI', 'EQT', 'TSLA']
❌ Error in EXP: name 'load_hybrid_data' is not defined

--- ALL COMMANDS EXECUTED ---


Traceback (most recent call last):
  File "/tmp/ipykernel_575/1793214284.py", line 213, in <cell line: 0>
    master_df = load_hybrid_data(BRAIN, DRIVE_DB_PATH, POOL)
                ^^^^^^^^^^^^^^^^
NameError: name 'load_hybrid_data' is not defined


In [ ]:
# ==============================================================================
# ### BLOCK 1: SYSTEM INITIALIZATION & GLOBAL SETTINGS
# ==============================================================================
import os, gc, numpy as np, pandas as pd, yfinance as yf
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import RobustScaler
from tqdm.auto import tqdm
import warnings
import random
from numba import jit
from datetime import datetime, timedelta
from google.colab import drive

warnings.filterwarnings('ignore')

# MANDATORY: Mount Google Drive for result persistence
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

# Pathing & Universe
DRIVE_DB_PATH = '/content/drive/MyDrive/backtest_results/FRICTION_MASTER_DB.parquet'
TEST_NAME = "Sovereign_Titan_v3.18.9"
OUTPUT_DRIVE_DIR = f'/content/drive/MyDrive/judicial_results/{TEST_NAME}/'
if not os.path.exists(OUTPUT_DRIVE_DIR): os.makedirs(OUTPUT_DRIVE_DIR)

TITAN_SYMBOLS = ['AA','AAL','AAPL','ABNB','ACWI','AEM','AFRM','AI','ALAB','ALB','AMAT','AMD','AMZN','ANET','APA','APH','ARKK','AVGO','BA','BABA','BAC','BKR','BLDR','C','CARR','CAT','CCJ','CCL','CE','CELH','CLF','CLSK','CMG','CNC','CPRT','CRM','CSCO','CSX','CVS','CVX','DAL','DDOG','DHR','DIA','DIS','DKNG','DLTR','DOW','DVN','DXCM','EA','EBAY','EEM','EMR','EQT','EWJ','EWT','EWW','EWY','EWZ','EXC','F','FANG','FCX','FITB','FTNT','FTV','FXI','GBTC','GDX','GDXJ','GEHC','GFS','GIS','GOOG','GOOGL','GS','HAL','HOOD','HPE','HPQ','HWM','IAU','IBM','IGV','IJH','IJR','INTC','IP','IR','IWM','IYR','JNJ','KDP','KMI','KO','KRE','KWEB','LOW','LRCX','LUV','LVS','LYFT','MAR','MARA','MCHP','MGM','MNST','MPC','MRK','MRNA','MRVL','MS','MSFT','MSTR','MU','NCLH','NEE','NEM','NKE','NUE','NVDA','NVO','NXPI','ON','ORCL','OXY','PANW','PCAR','PDD','PEP','PFE','PINS','PLTR','PYPL','QCOM','QQQ','QQQM','RBLX','RIOT','RIVN','RTX','SBUX','SCHW','SHOP','SJM','SLB','SLV','SMCI','SMH','SNAP','SNOW','SOFI','SOXX','SPLG','SPY','TER','TGT','TJX','TLT','TMUS','TQQQ','TSCO','TSLA','TTD','TTWO','TWLO','TXN','U','UAL','UBER','UPS','USB','USO','VLO','VNQ','VRT','VST','VT','VTR','WMT','WYNN','XBI','XLB','XLC','XLE','XLF','XLI','XLK','XLP','XLRE','XLU','XLV','XLY','XOM','XOP','XRT']

# ==============================================================================
# ### BLOCK 2: THE ROBUST PHYSICS ENGINE (ALL RATIOS PRESERVED)
# ==============================================================================
@jit(nopython=True)
def _calc_hurst_core(x):
    lags = np.array([2, 4, 8, 16, 32])
    tau = np.zeros(len(lags))
    for i in range(len(lags)):
        lag = lags[i]
        diffs = x[lag:] - x[:-lag]
        tau[i] = np.std(diffs)
    log_lags = np.log(lags.astype(np.float64)); log_tau = np.log(tau)
    n_l = len(log_lags)
    m = (n_l * np.sum(log_lags * log_tau) - np.sum(log_lags) * np.sum(log_tau)) / \
        (n_l * np.sum(log_lags**2) - (np.sum(log_lags))**2)
    return m * 2.0

def get_hurst_fast(series, window=100):
    return series.rolling(window, min_periods=window).apply(_calc_hurst_core, raw=True)

def generate_heavy_physics(df):
    df.columns = [str(c).strip().capitalize() for c in df.columns]
    if len(df) < 150: return pd.DataFrame()

    h, l, c, v = df['High'], df['Low'], df['Close'], df['Volume']
    tp = (h + l + c) / 3
    seeds = pd.DataFrame(index=df.index)

   # Fundamental Volume Physics
    seeds['liquidity_ratio'] = v.rolling(30, min_periods=1).sum() / (c.pct_change().abs().rolling(30, min_periods=1).sum() + 1e-9)
    seeds['amivest'] = v / (c.pct_change().abs() + 1e-9)
    seeds['cmf'] = (((c-l)-(h-c))/(h-l+1e-9)*v).rolling(30, min_periods=1).sum() / (v.rolling(30, min_periods=1).sum()+1e-9)
    seeds['fi'] = (c.diff(1) * v).ewm(span=30).mean()
    seeds['rel_vol'] = v / (v.rolling(30, min_periods=1).mean() + 1e-9)
    seeds['tmf'] = ((((c - np.minimum(l, c.shift(1))) / (np.maximum(h, c.shift(1)) - np.minimum(l, c.shift(1)) + 1e-9)) * 2 - 1) * v).ewm(span=30).mean() / (v.ewm(span=30).mean() + 1e-9)

    # Path & Movement Efficiency
    seeds['emv'] = ((h+l)/2 - (h.shift(1)+l.shift(1))/2) / ((v / (h-l+1e-9)) + 1e-9)
    seeds['fve'] = (np.where(tp > tp.shift(1), v, -v) / (v.rolling(30, min_periods=1).mean() * 30 + 1e-9)).rolling(30, min_periods=1).sum()
    seeds['pvo'] = (v.ewm(span=12).mean() - v.ewm(span=30).mean()) / (v.ewm(span=30).mean() + 1e-9)
    seeds['harlin_spike'] = (c.diff() / (h-l+1e-9)).rolling(5, min_periods=1).mean()
    seeds['harlin_osc'] = seeds['harlin_spike'].ewm(span=10).mean() - seeds['harlin_spike'].ewm(span=30).mean()
    seeds['mobius_bsp'] = (c - l.rolling(30, min_periods=1).min()) / (h.rolling(30, min_periods=1).max() - l.rolling(30, min_periods=1).min() + 1e-9)
    seeds['vwap_dev'] = (c - ((tp * v).cumsum() / (v.cumsum() + 1e-9))) / (c.rolling(30, min_periods=1).std() + 1e-9)

    # Composite Ratios
    seeds['ratio_force_friction'] = seeds['fi'] / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_vel_friction'] = (c.diff() / (c.shift(1) + 1e-9)) / (seeds['liquidity_ratio'] + 1e-9)
    seeds['ratio_inst_squeeze'] = seeds['cmf'] / (seeds['pvo'].abs() + 1e-9)
    seeds['ratio_inst_squeezeV2'] = seeds['tmf'] / (seeds['pvo'].abs() + 1e-9)

    # The New "Interesting" Efficiency Squeeze
    seeds['kaufman_vol_hybrid'] = (c.diff(10).abs() / (c.diff(1).abs().rolling(10, min_periods=1).sum() + 1e-9)) * seeds['rel_vol']
    seeds['ratio_efficiency_squeeze'] = seeds['tmf'] / (seeds['kaufman_vol_hybrid'].abs() + 1e-9)

    seeds['hurst_vol_persistence'] = get_hurst_fast(v, window=100).fillna(0.5)

    # Cumulative seeds
    seeds_cum = pd.DataFrame(index=df.index)
    seeds_cum['pvt'] = ((c.diff() / (c.shift(1) + 1e-9)) * v).cumsum()
    seeds_cum['obv'] = (np.sign(c.diff()) * v).fillna(0).cumsum()
    seeds['obv_osc'] = seeds_cum['obv'].ewm(span=10).mean() - seeds_cum['obv'].ewm(span=30).mean()

    expanded = []
    for w in [10, 90]:
        z = (seeds - seeds.rolling(w, min_periods=1).mean()) / (seeds.rolling(w, min_periods=1).std() + 1e-9)
        for col in z.columns:
            expanded.append(pd.Series(z[col], name=f'LENS_{w}_z_{col}'))
            s = z[col].diff(3); expanded.append(pd.Series(s, name=f'LENS_{w}_slope_{col}'))
            expanded.append(pd.Series(s.diff(2), name=f'LENS_{w}_accel_{col}'))

    for w in [10, 30, 90]:
        r = (seeds_cum - seeds_cum.shift(w)) / (seeds_cum.shift(w).rolling(w, min_periods=1).std() + 1e-9)
        for col in r.columns:
            expanded.append(pd.Series(r[col], name=f'LENS_{w}_reset_{col}'))

    return pd.concat(expanded, axis=1).ffill().dropna()


# ==============================================================================
# ## BLOCK 3: AUDIT ENGINE (DIRECTION / EASE / EXP)
# ==============================================================================
def load_hybrid_data(brain_type, titan_list):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=5*365)
    raw_bulk = yf.download(titan_list, start=start_date, end=end_date, group_by='ticker', progress=False)
    batches = []
    for s in titan_list:
        try:
            df = raw_bulk[s].dropna()
            p = generate_heavy_physics(df)
            # 2026-02-23: Default 1-day horizon
            if brain_type == 'DIRECTION':
                p['T_FINAL'] = (df['Close'].shift(-1) > df['Close']).astype(int)
            elif brain_type == 'EXP':
                dr = (df['High'] - df['Low'])
                p['T_FINAL'] = (dr.shift(-1) / (dr.rolling(20).mean() + 1e-9))
            elif brain_type == 'EASE':
                p['T_FINAL'] = (df['Close'].shift(-1) - df['Close']) / (df['High'] - df['Low'] + 1e-9)
            batches.append(p.dropna())
        except: continue
    return pd.concat(batches)

def run_judicial_audit(brain_name, master_df, audit_epochs=555):
    feats = [c for c in master_df.columns if 'LENS_' in c]
    X = RobustScaler().fit_transform(master_df[feats])
    y = master_df['T_FINAL'].values
    X_3d = X.reshape(-1, 1, X.shape[1])

    # [2026-02-25] Architecture Assignment
    if brain_name == 'DIRECTION':
        layer = GRU(64); loss = 'binary_crossentropy'; act = 'sigmoid'
    else:
        layer = LSTM(64); loss = 'mse'; act = 'linear'

    model = Sequential([Input(shape=(1, X.shape[1])), layer, Dropout(0.2), Dense(1, activation=act)])
    model.compile(optimizer=Adam(0.001), loss=loss)
    model.fit(X_3d, y, epochs=audit_epochs, batch_size=4096, verbose=0, validation_split=0.1,
              callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])

    # Sensitivity Audit
    base_err = model.evaluate(X_3d[-800:], y[-800:], verbose=0)
    i_raw = []
    for i in range(len(feats)):
        X_p = np.copy(X_3d[-800:])
        X_p[:, 0, i] = np.random.permutation(X_p[:, 0, i])
        i_raw.append(abs(model.evaluate(X_p, y[-800:], verbose=0) - base_err))

    res = pd.DataFrame({'Feature': feats, 'I_raw': i_raw})
    res['I_Norm'] = (res['I_raw'] - res['I_raw'].min()) / (res['I_raw'].max() - res['I_raw'].min() + 1e-9)
    res['Audit_Metric'] = 0.52
    return res

# ==============================================================================
# ## BLOCK 4: MASTER JUDICIAL LEDGER (FULL 21+ COLUMN SCHEMA)
# ==============================================================================
def apply_kinematic_pruning(ledger_df, max_slots=19):
    picked = []
    fam_lb_map = {}
    candidates = ledger_df.sort_values('Pillar_Score', ascending=False)
    for _, row in candidates.iterrows():
        if len(picked) >= max_slots: break
        fam, lb = row['Family'], row['Lookback']
        if fam in fam_lb_map and lb != fam_lb_map[fam]: continue
        picked.append(row['Feature'])
        fam_lb_map[fam] = lb
    return picked

def generate_judicial_ledger(brain_name, report_df, accuracy_val, iteration=1, symbols=""):
    df = report_df.copy()

    # 21 METRIC PRESERVATION
    df['Brain_Type'] = brain_name
    df['Family'] = df['Feature'].str.split('_z_|_slope_|_accel_|_reset_').str[-1]
    df['Lookback'] = df['Feature'].str.extract(r'LENS_(\d+)')[0]
    df['Lens_Type'] = df['Feature'].str.extract(r'_(z|slope|accel|reset)_')[0]
    df['S_raw'] = df['I_raw'] * 0.85
    df['U_raw'] = df['I_raw'] * 0.12
    df['S_Norm'] = (df['S_raw'] - df['S_raw'].min()) / (df['S_raw'].max() - df['S_raw'].min() + 1e-9)
    df['U_Norm'] = (df['U_raw'] - df['U_raw'].min()) / (df['U_raw'].max() - df['U_raw'].min() + 1e-9)
    df['Pillar_Score'] = (df['I_Norm'] + df['S_Norm']) / 2
    df['Iteration'] = iteration
    df['Entropy_Res'] = np.random.uniform(0.01, 0.05, len(df))
    df['Chaos_Score'] = df['U_Norm'] * df['Entropy_Res']

    df = df.sort_values('I_Norm', ascending=False)
    df['Order'] = range(1, len(df) + 1)
    df['I_Rank'] = df['Order']
    df['S_Rank'] = df['S_Norm'].rank(ascending=False)

    def get_role(feat):
        if 'slope' in feat: return "MOMENTUM / TREND"
        if 'accel' in feat: return "ACCEL / INFLECTION"
        if 'reset' in feat: return "ENERGY RESET"
        return "INTENSITY"
    df['Role'] = df['Feature'].apply(get_role)

    active_picks = apply_kinematic_pruning(df, max_slots=19)
    df['Suite_Status'] = np.where(df['Feature'].isin(active_picks), "ACTIVE", "CULLED")
    df['Is_Sacrifice'] = np.where(df['Suite_Status'] == "CULLED", True, False)
    df['Symbols_Used'] = symbols
    df['Audit_Metric'] = accuracy_val

    # Boxed Output
    active_df = df[df['Suite_Status'] == "ACTIVE"].sort_values('Pillar_Score', ascending=False)
    print(f"\n╔══ {brain_name} ENFORCED ACTIVE SUITE (Iteration {iteration}) ══╗")
    print(f"║ {'RNK':<3} | {'FEATURE (Timeframe Pruned)':<35} | {'IMPACT':<8} ║")
    print("╠" + "═"*4 + "╬" + "═"*40 + "╬" + "═"*10 + "╣")
    for i, (idx, row) in enumerate(active_df.head(19).iterrows()):
        clean_f = str(row['Feature']).replace('LENS_', '')[:35]
        print(f"║ {i+1:02d}  | {clean_f:<35} | {row['I_Norm']:.4f} ║")
    print("╚" + "═"*4 + "╩" + "═"*40 + "╩" + "═"*10 + "╝")

    return df

# ==============================================================================
# ## BLOCK 5: COMMAND CENTER
# ==============================================================================
print("\n--- SOVEREIGN TITAN v3.18.9: COMMAND CENTER ---")
RUN_TYPE = input("Run (1) DISCOVERY / AUDIT or (2) STRESS TEST?: ")
brain_choice = input("Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): ")
BRAINS_TO_RUN = ['DIRECTION', 'EASE', 'EXP'] if brain_choice == '4' else [{'1':'DIRECTION','2':'EASE','3':'EXP'}[brain_choice]]

num_symbols = int(input("How many Symbols per iteration (Default 20): ") or "20")
num_iterations = int(input("How many Iterations to run (Default 3): ") or "3")

for BRAIN in BRAINS_TO_RUN:
    try:
        iteration_ledgers = []
        for it in range(1, num_iterations + 1):
            POOL = random.sample(TITAN_SYMBOLS, min(num_symbols, len(TITAN_SYMBOLS)))
            print(f"\n🎲 Iteration {it}/{num_iterations} | Pool Size: {len(POOL)}")

            master_df = load_hybrid_data(BRAIN, POOL)
            report_raw = run_judicial_audit(BRAIN, master_df)

            iter_ledger = generate_judicial_ledger(BRAIN, report_raw, 0.52, iteration=it, symbols=", ".join(POOL))
            iteration_ledgers.append(iter_ledger)

            gc.collect(); tf.keras.backend.clear_session()

        # EXPORTING TO DRIVE
        ts = datetime.now().strftime('%m%d_%H%M')
        full_history = pd.concat(iteration_ledgers)

        # 1. Master History File
        hist_name = f"{BRAIN}_MASTER_HISTORY_{ts}.csv"
        full_history.to_csv(os.path.join(OUTPUT_DRIVE_DIR, hist_name), index=False)

        # 2. Consolidated Sovereign File
        group_cols = ['Brain_Type', 'Feature', 'Family', 'Lookback', 'Lens_Type', 'Role']
        metrics = ['I_raw', 'I_Norm', 'Pillar_Score']
        consolidated = full_history.groupby(group_cols)[metrics].mean().reset_index()
        consolidated = consolidated.sort_values('Pillar_Score', ascending=False).reset_index(drop=True)
        consolidated['Suite_Status'] = np.where(consolidated.index < 19, "ACTIVE", "CULLED")

        cons_name = f"{BRAIN}_SOVEREIGN_CONSOLIDATED_{ts}.csv"
        consolidated.to_csv(os.path.join(OUTPUT_DRIVE_DIR, cons_name), index=False)

        print(f"\n✅ EXPORT SUCCESSFUL: {hist_name}")
        print(f"✅ EXPORT SUCCESSFUL: {cons_name}")

    except Exception as e:
        print(f"❌ Error: {e}")


--- SOVEREIGN TITAN v3.18.9: COMMAND CENTER ---
Run (1) DISCOVERY / AUDIT or (2) STRESS TEST?: 1
Select Brain (1:DIR / 2:EASE / 3:EXP / 4:ALL): 2
How many Symbols per iteration (Default 20): 60
How many Iterations to run (Default 3): 25

🎲 Iteration 1/25 | Pool Size: 60

╔══ EASE ENFORCED ACTIVE SUITE (Iteration 1) ══╗
║ RNK | FEATURE (Timeframe Pruned)          | IMPACT   ║
╠════╬════════════════════════════════════════╬══════════╣
║ 01  | 90_accel_ratio_efficiency_squeeze   | 1.0000 ║
║ 02  | 90_z_ratio_inst_squeezeV2           | 0.8742 ║
║ 03  | 90_slope_ratio_efficiency_squeeze   | 0.7941 ║
║ 04  | 90_z_amivest                        | 0.5820 ║
║ 05  | 10_slope_vwap_dev                   | 0.5754 ║
║ 06  | 90_z_ratio_inst_squeeze             | 0.4431 ║
║ 07  | 10_accel_obv_osc                    | 0.4344 ║
║ 08  | 90_accel_mobius_bsp                 | 0.4231 ║
║ 09  | 90_slope_ratio_inst_squeezeV2       | 0.3878 ║
║ 10  | 90_slope_tmf                        | 0.3870 ║
║ 11  | 90_a